# `ap-2026-q-2` · Notebook principal (versión liviana e independiente)
Predice `condition` (`new` / `used`) de publicaciones de MercadoLibre. Métrica: **accuracy**.

**Qué cambió respecto de la versión larga**
- Sin gráficos, EDA ni pruebas intermedias: solo el feature engineering y los modelos que se usan.
- Cada parte lenta tiene un **interruptor `True/False`** (celda 1) y guarda su resultado en una carpeta de **cache**. La segunda vez que corrés todo, lo que no cambió se *carga del disco* en segundos.
- Es **independiente**: carga los datos, arma las features, entrena y genera el `submission.csv`. No necesita ningún otro notebook.

**Cómo trabajar**
| Situación | Configuración |
|---|---|
| Probar una idea rápido | `FAST_MODE = True` y `SAMPLE_TRAIN = 20000` |
| Corrida completa / submission final | `FAST_MODE = False` y `SAMPLE_TRAIN = None` |
| Probar los hiperparámetros / las features nuevas | `TUNE_LGB`, `F_SELLER_DEV`, `F_PAIR_TE` (de a uno) y mirar la sección 6b y la 10 |
| Cambiaste el código de una feature | `FORCE_RECOMPUTE = True` **una vez** (o borrá la carpeta `cache_ap`). Las listas de palabras clave y las predicciones de los modelos se invalidan solas |

Orden del notebook: 1) configuración · 2) datos · 3) limpieza · 4) features base · 5) features de texto y bloques opcionales · 6) modelos · 7) mezcla y submission.

## 1. Configuración (acá se prende y apaga todo)
Los interruptores de **features** controlan qué columnas se agregan; los de **modelos**, qué se entrena. Los valores por defecto reproducen la mejor versión que tenías (ensamble LightGBM + XGBoost + CatBoost).

In [1]:
# ======================= INTERRUPTORES: poné True / False =======================
FAST_MODE        = False   # True = pruebas rápidas: learning rate alto, 1 semilla y sin CatBoost (para iterar features)
SAMPLE_TRAIN     = None    # ej. 20000 = usa solo esas filas del train. SOLO para depurar: no sirve para el submission final
USE_CACHE        = True    # guarda en disco los resultados lentos y los reutiliza en la próxima corrida
FORCE_RECOMPUTE  = False   # True = ignora el cache y recalcula todo (usalo una vez si cambiaste código de una feature)
CACHE_DIR        = 'cache_ap'

# --- Features (cada una se calcula una sola vez y queda en cache) ---
F_TEXT_OOF            = True    # 8 modelos de texto (TF-IDF + reg. logística) con predicciones OOF        [lento]
F_KNN_TITLE           = True    # etiquetas de los títulos más parecidos del train                          [lento]
F_TIME_NEIGHBORS      = True    # etiqueta de las publicaciones vecinas EN EL TIEMPO del mismo vendedor
F_LEGACY_TEXT_VIEWS   = False   # 3 vistas de texto antiguas, redundantes con F_TEXT_OOF (True = reproduce el CV original)
F_SIBLINGS            = False   # predicción promedio de los "hermanos" del vendedor (en tus pruebas no mejoró el CV)
F_WITHIN_SELLER_TITLE = True    # vecino más parecido por título dentro del vendedor (en la última ablación: -0,0008 al quitarlo)
F_SELLER_DEV          = True    # cuánto se desvía cada publicación de lo habitual de su vendedor (foto, precio, garantía...). Sin usar el target. Junto con F_PAIR_TE: -0,0019 al quitarlos
F_PAIR_TE             = True    # target encoding de pares: vendedor×tamaño de foto, vendedor×precio relativo, categoría×garantía...
F_TEXT_CROSS         = False   # texto CRUZADO con la categoría (en tu prueba no mejoró: el CV subió +0,0003 al quitarlo)
F_SELLER_SIM         = False   # (NUEVO) tasa de 'used' de los vendedores PARECIDOS (por categorías, vocabulario de títulos y estilo). Apunta a los vendedores sin historial
F_CLUSTER           = False   # (NUEVO) agrupa las publicaciones por TIPO DE PRODUCTO (KMeans sobre los títulos) y calcula tasas de 'used' por grupo, grupo×foto, grupo×garantía y vendedor×grupo
CLUSTER_KS          = (60, 300)   # cantidad de grupos: uno grueso y otro fino
F_SVD_TITLE         = False   # (NUEVO) 50 componentes SVD del TF-IDF del título como variables numéricas (temas del título)
SVD_DIM             = 50      # cantidad de componentes SVD
F_TEXT_NBSVM        = False   # (NUEVO) NB-SVM: regresión logística sobre TF-IDF reponderado por Naive Bayes (clasificación bayesiana + lineal); OOF por vista

# --- Modelos ---
RUN_LGB     = True;  N_SEEDS_LGB = 6       # LightGBM, promedia N semillas (hasta 6; más semillas = predicciones más estables)
RUN_XGB     = True;  N_SEEDS_XGB = 3       # XGBoost, promedia N semillas (hasta 5)
RUN_CAT     = True;  N_SEEDS_CAT = 3       # CatBoost (el más lento), promedia N semillas (hasta 5)
BLEND       = 'meta'                       # 'meta' = regresión logística sobre las predicciones | 'mean' = promedio simple
NN_DIR      = None                         # opcional: carpeta con nn_feats_train.csv y nn_feats_test.csv de la red neuronal
BERT_DIR        = '/kaggle/input/notebooks/mirandacontieri/notebook-xlm-roberta-base'                      # opcional: carpeta con bert_feats_train.csv y bert_feats_test.csv (del notebook de BERT)
BERT_AS_FEATURE = True                     # True = suma 'bert_oof' como variable de los modelos (lo más probable que ayude)
BERT_IN_BLEND   = False                    # True = además lo suma como modelo en la mezcla final
BERT_EXTRA_DIRS  =["/kaggle/input/notebooks/mirandacontieri/notebook-roberta-base-bne-v5-ap-2026",'/kaggle/input/notebooks/mirandacontieri/notebook-bert-titulos-ap']                      # otras corridas de BERT (p. ej. con USE_META_TEXT o con otro modelo): lista de carpetas. Cada una suma su propia variable
SHOW_DIAGNOSTICS = True                   # True = imprime accuracy por tipo de vendedor al final

# --- Ajuste de hiperparámetros y ablación (opcionales, lentos: usar de a uno) ---
TUNE_LGB         = False   # True = busca mejores hiperparámetros de LightGBM con Optuna (guarda el resultado en cache_ap/tuned_lgb.json)
N_TRIALS         = 30      # cantidad máxima de pruebas de Optuna
TUNE_TIMEOUT_MIN = 45      # tiempo máximo del ajuste, en minutos
USE_TUNED_PARAMS = False   # True = LightGBM usa los parámetros guardados en tuned_lgb.json
RUN_ABLATION     = True   # True = mide cuánto aporta cada bloque de features (reentrena sin cada uno); sirve para limpiar lo que no ayuda
ABL_SEEDS        = (1, 2)  # semillas que promedia cada medición de la ablación
ABL_SKIP_SINGLE  = True   # True = salta la ablación bloque por bloque y mide solo los GRUPOS de abajo (más rápido)
ABL_GROUPS       = {'vecinos (knn_title + time_neighbors)': ['knn_title', 'time_neighbors'],       # grupos de bloques que se quitan JUNTOS
                    'vendedor nuevos (seller_dev + pair_te)': ['seller_dev', 'pair_te'],
                    'todos los BERT': ['bert', 'bert2', 'bert3'],
                    'todos vendedor/vecinos': ['knn_title', 'time_neighbors', 'seller_dev', 'pair_te']}
RUN_LAB         = False          # True = ejecuta la sección 11 (laboratorio de iteraciones)
RUN_INSPECTOR   = False          # True = activa la sección 12 (inspector de errores del TRAIN y probador de reglas con control de sobreajuste)
UNC_BAND        = (0.15, 0.85)   # banda "dudosa": probabilidades entre estos valores = el modelo no está seguro
LAB_MIN_DELTA   = 0.0003         # mejora mínima de CV (accuracy OOF) para recomendar enviar un candidato
LAB_MIN_Z       = 2.0            # significancia mínima de esa mejora (filas que mejoran vs. empeoran; prueba de McNemar)
REF_SUBMISSION  = None           # ruta del CSV de tu último envío (para contar cuántas filas del test cambian respecto de él)
REF_RUN         = None           # corrida de referencia, ej. '20261024_1359': todo se compara contra esa corrida (usa laboratorio/<REF_RUN>_base_oof.npy)
LAB_DIR         = 'laboratorio'
CAT_EVAL_METRIC  = 'Logloss'  # métrica con la que CatBoost elige cuándo parar. Probá 'Logloss' (menos ruidosa que el accuracy)
SAVE_COPY        = True        # además de submission.csv, guarda una copia con el CV y la hora en el nombre (para no perder candidatos)

# --- Ajustes generales ---
TARGET = 'condition'; RANDOM_STATE = 42; N_FOLDS = 5
DATA_DIR = None                  # opcional: carpeta con train_data.jsonlines y test_data.jsonlines (None = los busca solo)
DOWNLOAD_WITH_KAGGLEHUB = True   # si no encuentra los archivos, los baja con kagglehub (necesita sesión de Kaggle)

# Valores derivados de FAST_MODE (no tocar)
LGB_LR   = 0.08 if FAST_MODE else 0.03
SEEDS    = [1, 2, 3, 4, 5, 6][:1 if FAST_MODE else max(1, min(N_SEEDS_LGB, 6))]
RUN_CAT_EFF = RUN_CAT and not FAST_MODE

## 2. Datos y utilidades de cache
Carga `train` y `test` (busca en `/kaggle/input`, en el cache de kagglehub o los descarga), define el target (`used` = 1) y una *firma* de los datos: si cambia el train, el target o la semilla, el cache viejo se ignora solo.

In [2]:
import os, re, hashlib, time, warnings, unicodedata
import numpy as np, pandas as pd
from scipy import sparse
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB

os.makedirs(CACHE_DIR, exist_ok=True)
def md5(*parts):
    h = hashlib.md5()
    for p in parts: h.update(p if isinstance(p, (bytes, bytearray)) else str(p).encode())
    return h.hexdigest()
def cache_load(name):
    p = os.path.join(CACHE_DIR, name + '.pkl')
    return pd.read_pickle(p) if (USE_CACHE and not FORCE_RECOMPUTE and os.path.exists(p)) else None
def cache_save(name, obj):
    if USE_CACHE: pd.to_pickle(obj, os.path.join(CACHE_DIR, name + '.pkl'))

def list_files():
    dirs = [DATA_DIR] if DATA_DIR else ['/kaggle/input', os.path.expanduser('~/.cache/kagglehub/competitions')]
    return [os.path.join(r, f) for d in dirs if d and os.path.isdir(d) for r, _, fs in os.walk(d) for f in fs if f.endswith('.jsonlines')]
paths = list_files()
if not paths and DOWNLOAD_WITH_KAGGLEHUB:
    import kagglehub
    base = kagglehub.competition_download('ap-2026-q-2')
    paths = [os.path.join(r, f) for r, _, fs in os.walk(base) for f in fs if f.endswith('.jsonlines')]
pick = lambda w: [p for p in paths if w in os.path.basename(p).lower()][0]
print('train:', pick('train')); print('test :', pick('test'))

train = pd.read_json(pick('train'), lines=True); test = pd.read_json(pick('test'), lines=True)
assert TARGET in train.columns, f'el train no tiene la columna {TARGET}'
if SAMPLE_TRAIN: train = train.sample(SAMPLE_TRAIN, random_state=RANDOM_STATE)
train = train.reset_index(drop=True); test = test.reset_index(drop=True)
ntr, nte = len(train), len(test)

classes = sorted(train[TARGET].unique())
yv = pd.Series((train[TARGET] == classes[-1]).astype(int).values)         # 'used' = 1
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)    # los MISMOS folds en todo el notebook
DATA_SIG = md5(ntr, nte, SAMPLE_TRAIN, RANDOM_STATE, N_FOLDS, yv.values.tobytes())
print(f'train {train.shape} | test {test.shape} | % {classes[-1]}: {yv.mean():.3f}')

train: /kaggle/input/competitions/ap-2026-q-2/train_data.jsonlines
test : /kaggle/input/competitions/ap-2026-q-2/test_data.jsonlines
train (70000, 45) | test (30000, 44) | % used: 0.463


## 3. Aplanado, faltantes y outliers
- **Aplanado:** convierte las columnas anidadas (`pictures`, `attributes`, `shipping`, `seller_address`, `tags`...) en variables simples.
- **Faltantes:** numéricas con la mediana del train (+ indicador `_isna`), categóricas como `'missing'`, y flags de garantía.
- **Outliers:** `log1p` de precios y cantidades y recorte (winsorizing) a los percentiles 0,5 % / 99,5 %. No se eliminan filas.

El resultado (`df_o` para train, `te_o` para test) queda en cache.

In [3]:
def g(d, *keys):
    for k in keys:
        if isinstance(d, dict): d = d.get(k)
        else: return None
    return d
def to_dt(s):
    if pd.api.types.is_numeric_dtype(s): return pd.to_datetime(s, unit='ms', utc=True, errors='coerce')
    return pd.to_datetime(s, utc=True, errors='coerce')
def to_seconds(s):
    return (to_dt(s) - pd.Timestamp(0, tz='UTC')).dt.total_seconds().values

def flatten(df):
    o = pd.DataFrame(index=df.index)
    for c in ['price','base_price','initial_quantity','available_quantity','sold_quantity','seller_id',
              'listing_type_id','buying_mode','currency_id','status','category_id','accepts_mercadopago','automatic_relist']:
        o[c] = df[c]
    o['original_price_isna'] = df['original_price'].isna().astype(int)
    o['official_store'] = df['official_store_id'].notna().astype(int)
    o['has_video'] = df['video_id'].notna().astype(int)
    o['has_catalog'] = df['catalog_product_id'].notna().astype(int)
    o['has_parent'] = df['parent_item_id'].notna().astype(int)
    o['has_subtitle'] = df['subtitle'].notna().astype(int)
    o['state'] = df['seller_address'].apply(lambda d: g(d,'state','name'))
    o['city']  = df['seller_address'].apply(lambda d: g(d,'city','name'))
    o['free_shipping'] = df['shipping'].apply(lambda d: g(d,'free_shipping')).astype(float)
    o['local_pick_up'] = df['shipping'].apply(lambda d: g(d,'local_pick_up')).astype(float)
    o['shipping_mode'] = df['shipping'].apply(lambda d: g(d,'mode'))
    o['n_ship_methods'] = df['shipping'].apply(lambda d: len(g(d,'methods') or []))
    for name, col in [('n_pictures','pictures'), ('n_attributes','attributes'), ('n_variations','variations'), ('n_tags','tags'),
                      ('n_payment_methods','non_mercado_pago_payment_methods'), ('n_sub_status','sub_status')]:
        o[name] = df[col].apply(lambda x: len(x) if isinstance(x, list) else 0)
    tags_exp = df['tags'].apply(lambda x: x if isinstance(x, list) else [])
    for t in ['dragged_bids_and_visits','good_quality_thumbnail','poor_quality_thumbnail','free_relist','dragged_visits']:
        o['tag_'+t] = tags_exp.apply(lambda l: int(t in l))
    o['title'] = df['title'].fillna('')
    o['title_len'] = o['title'].str.len(); o['title_words'] = o['title'].str.split().str.len()
    o['title_upper_ratio'] = o['title'].apply(lambda s: sum(ch.isupper() for ch in s)/max(len(s),1))
    o['warranty'] = df['warranty'].fillna('').astype(str).str.lower().str.strip()
    dc, lu, st, et = to_dt(df['date_created']), to_dt(df['last_updated']), to_dt(df['start_time']), to_dt(df['stop_time'])
    o['created_year'] = dc.dt.year; o['created_month'] = dc.dt.month; o['created_dow'] = dc.dt.dayofweek
    o['days_active'] = (et - st).dt.total_seconds()/86400
    o['days_since_update'] = (lu - dc).dt.total_seconds()/86400
    o['attr_names'] = df['attributes'].apply(lambda l: ' | '.join(sorted({a.get('name','') for a in l})) if isinstance(l, list) else '')
    if TARGET in df.columns: o[TARGET] = df[TARGET]
    return o

def clean_missing(d, medians=None):
    d = d.copy()
    d['warranty_missing'] = (d['warranty'] == '').astype(int)
    d['warranty_has_no']  = d['warranty'].str.contains('sin garant', na=False).astype(int)
    d['warranty_yes']     = d['warranty'].str.contains(r'^(si|sí)$|meses|año|a\u00f1o|fabrica|f\u00e1brica|oficial', regex=True, na=False).astype(int)
    skip = {'title', 'attr_names', 'warranty', TARGET}
    cat = [c for c in d.columns if c not in skip and not pd.api.types.is_numeric_dtype(d[c])]      # texto/categorías (los bool quedan como están)
    d[cat] = d[cat].fillna('missing')
    num = d.select_dtypes(include=np.number).columns.difference([TARGET])
    if medians is None: medians = d[num].median()
    for c in num:
        if d[c].isna().any(): d[c+'_isna'] = d[c].isna().astype(int)
    d[num] = d[num].fillna(medians)
    return d, medians

LOG_COLS = ['price','base_price','initial_quantity','available_quantity','sold_quantity']
def treat_outliers(d, bounds=None):
    d = d.copy()
    for c in LOG_COLS: d[c+'_log'] = np.log1p(d[c].clip(lower=0))
    clip_cols = [c+'_log' for c in LOG_COLS] + ['title_len','n_pictures','n_attributes','days_active','days_since_update']
    if bounds is None: bounds = {c: (d[c].quantile(.005), d[c].quantile(.995)) for c in clip_cols}
    for c,(lo,hi) in bounds.items(): d[c] = d[c].clip(lo,hi)
    return d, bounds

frames = cache_load(f'frames_{DATA_SIG}')
if frames is None:
    t0 = time.time()
    df_c, MED = clean_missing(flatten(train)); df_o, BOUNDS = treat_outliers(df_c)
    te_c, _   = clean_missing(flatten(test), medians=MED); te_o, _ = treat_outliers(te_c, bounds=BOUNDS)
    for d in (df_o, te_o): d['seller_cat'] = d['seller_id'].astype(str) + '_' + d['category_id'].astype(str)
    cache_save(f'frames_{DATA_SIG}', (df_o, te_o)); print(f'aplanado y limpieza: {time.time()-t0:.0f}s')
else:
    df_o, te_o = frames; print('aplanado y limpieza: cargado del cache')
print(df_o.shape, te_o.shape)

aplanado y limpieza: 4s
(70000, 57) (30000, 56)


## 4. Features base
- **Tabulares:** variables numéricas, categorías de baja cardinalidad en *one-hot* y frecuencia de las de alta cardinalidad.
- **Vendedor:** cantidad de publicaciones y precio medio/desvío del vendedor (train + test juntos, sin usar el target).
- **Target encoding OOF** de `seller_id`, `category_id`, `city` y `seller_cat`: la tasa de `used` de cada grupo, calculada solo con los *otros* folds para no filtrar la etiqueta.
- **Precio relativo** a su categoría y a su vendedor, y fracción vendida.

In [4]:
NUM = ['price_log','base_price_log','initial_quantity_log','available_quantity_log','sold_quantity_log',
       'title_len','title_words','title_upper_ratio','n_pictures','n_attributes','n_variations','n_tags','n_payment_methods',
       'n_ship_methods','days_active','days_since_update','created_year','created_month','created_dow',
       'free_shipping','local_pick_up','accepts_mercadopago','automatic_relist','official_store','has_video','has_catalog',
       'has_parent','has_subtitle','original_price_isna','warranty_missing','warranty_has_no','warranty_yes'] + \
      [c for c in df_o.columns if c.startswith('tag_')]
NUM = [c for c in NUM if c in df_o.columns]
LOWCAT = ['listing_type_id','buying_mode','status','currency_id','shipping_mode','state']
HIGHCAT = ['category_id','seller_id','city']

def freq_enc(train_s, s):
    return s.map(train_s.value_counts(normalize=True)).fillna(0)
def base_X(d):
    Z = d[NUM + LOWCAT].copy().reset_index(drop=True)
    for c in HIGHCAT: Z[c+'_freq'] = freq_enc(df_o[c], d[c]).reset_index(drop=True)
    return pd.get_dummies(Z, columns=LOWCAT)
Xa = base_X(df_o); Xb = base_X(te_o).reindex(columns=Xa.columns, fill_value=0)

# agregados por vendedor (train + test juntos, no usan el target)
allc = pd.concat([df_o[['seller_id','price_log']], te_o[['seller_id','price_log']]])
agg = allc.groupby('seller_id')['price_log'].agg(['count','mean','std'])
for d, Z in [(df_o, Xa), (te_o, Xb)]:
    for s in ['count','mean','std']: Z['seller_'+s] = d['seller_id'].map(agg[s]).fillna(0).values

# target encoding OOF con suavizado
def target_encode(col_tr, col_te, smooth=10):
    prior = yv.mean(); oof = np.zeros(len(col_tr)); col_tr = col_tr.reset_index(drop=True)
    for tr_i, va_i in skf.split(np.zeros(len(col_tr)), yv):
        s = pd.DataFrame({'c': col_tr.iloc[tr_i].values, 'y': yv.iloc[tr_i].values}).groupby('c')['y'].agg(['sum','count'])
        enc = (s['sum'] + prior*smooth) / (s['count'] + smooth)
        oof[va_i] = col_tr.iloc[va_i].map(enc).fillna(prior).values
    s = pd.DataFrame({'c': col_tr.values, 'y': yv.values}).groupby('c')['y'].agg(['sum','count'])
    enc = (s['sum'] + prior*smooth) / (s['count'] + smooth)
    return oof, col_te.reset_index(drop=True).map(enc).fillna(prior).values
for c in ['seller_id', 'category_id', 'city', 'seller_cat']:
    Xa[c+'_te'], Xb[c+'_te'] = target_encode(df_o[c], te_o[c])

# precio relativo a la categoría / al vendedor (train + test juntos)
cols = ['category_id', 'seller_id', 'price_log']
comb = pd.concat([df_o[cols], te_o[cols]], ignore_index=True)
g_cat = comb.groupby('category_id')['price_log']
comb['price_rel_cat']    = comb['price_log'] - g_cat.transform('median')
comb['price_z_cat']      = comb['price_rel_cat'] / (g_cat.transform('std').fillna(0) + 1e-6)
comb['price_rank_cat']   = g_cat.rank(pct=True)
comb['cat_n']            = g_cat.transform('count')
comb['price_rel_seller'] = comb['price_log'] - comb.groupby('seller_id')['price_log'].transform('mean')
for c in ['price_rel_cat', 'price_z_cat', 'price_rank_cat', 'cat_n', 'price_rel_seller']:
    Xa[c] = comb[c].values[:ntr]; Xb[c] = comb[c].values[ntr:]
Xa['sold_frac'] = (df_o['sold_quantity'] / (df_o['initial_quantity'] + 1)).values
Xb['sold_frac'] = (te_o['sold_quantity'] / (te_o['initial_quantity'] + 1)).values
print('features base:', Xa.shape)

features base: (70000, 98)


## 5. Funciones de features de texto
Definiciones (no calculan nada todavía):
- **`build_sources`:** separa el texto en "fuentes" normalizadas (sin acentos ni mayúsculas): título, garantía, atributos, variaciones, medios de pago, tamaños de foto y etiquetas.
- **`text_numeric_feats`:** estilo del título (dígitos, mayúsculas, códigos de modelo, unidades...) y señales de las fotos (tamaño, relación de aspecto, antigüedad de la foto).
- **`kw_counts`:** cuántas palabras "de usado" o "de nuevo" aparecen en el título y la garantía.
- **`get_mats`:** matrices TF-IDF de cada fuente (se calculan una sola vez por sesión y se comparten entre bloques).

In [5]:
def _norm(s):
    s = unicodedata.normalize('NFKD', str(s).lower()); return ''.join(ch for ch in s if not unicodedata.combining(ch))
def _tok(x): return re.sub(r'[^a-z0-9]+', '_', _norm(x)).strip('_')
L = lambda v: v if isinstance(v, list) else []

def build_sources(raw):
    raw = raw.reset_index(drop=True); S = pd.DataFrame(index=raw.index)
    S['title']    = raw['title'].fillna('').map(_norm)
    S['warranty'] = raw['warranty'].fillna('sin_dato').map(_norm)
    S['attr_ids'] = raw['attributes'].map(lambda v: ' '.join(_tok(a.get('id', '')) for a in L(v)))
    S['attr_vals'] = raw['attributes'].map(lambda v: ' '.join(f"{_tok(a.get('id',''))}__{_tok(a.get('value_name',''))}" for a in L(v)))
    S['variations'] = raw['variations'].map(lambda v: ' '.join(f"{_tok(c.get('name',''))}__{_tok(c.get('value_name',''))}"
                                                              for x in L(v) for c in L(x.get('attribute_combinations'))))
    S['payments'] = raw['non_mercado_pago_payment_methods'].map(lambda v: ' '.join(_tok(p.get('description', '')) for p in L(v)))
    S['pics'] = raw['pictures'].map(lambda v: ' '.join(f"sz_{p.get('size','')} mx_{p.get('max_size','')}" for p in L(v)))
    dom = raw['permalink'].fillna('').str.extract(r'https?://([^./]+)\.')[0].fillna('none')
    S['meta'] = (raw['tags'].map(lambda v: ' '.join(_tok(t) for t in L(v))) + ' dom_' + dom
                 + ' ' + raw['sub_status'].map(lambda v: ' '.join('ss_' + _tok(t) for t in L(v))))
    return S

_MEM = {}
def get_sources():
    if 'S' not in _MEM:
        got = cache_load(f'sources_{DATA_SIG}')
        if got is None: got = (build_sources(train), build_sources(test)); cache_save(f'sources_{DATA_SIG}', got)
        _MEM['S'] = got
    return _MEM['S']

def text_numeric_feats(raw):
    raw = raw.reset_index(drop=True); t = raw['title'].fillna(''); F = pd.DataFrame(index=raw.index)
    F['t_digits'] = t.str.count(r'\d'); F['t_digit_ratio'] = F['t_digits'] / (t.str.len() + 1)
    F['t_has_year'] = t.str.contains(r'\b(19|20)\d{2}\b').astype(int)
    F['t_model_code'] = t.str.count(r'\b(?=\w*\d)(?=\w*[A-Za-z])\w{3,}\b')
    F['t_caps_words'] = t.str.count(r'\b[A-Z]{3,}\b')
    F['t_all_lower'] = (t == t.str.lower()).astype(int); F['t_title_case'] = t.str.istitle().astype(int)
    F['t_punct'] = t.str.count(r'[!¡?\*\-/()\.,]')
    F['t_has_money'] = t.str.contains(r'\$|usd|u\$s', case=False).astype(int)
    F['t_has_qty'] = t.str.contains(r'\b(x\s?\d+|\d+\s?(un|u|unid|unidades|pcs|piezas))\b', case=False).astype(int)
    F['t_has_unit'] = t.str.contains(r'\b\d+\s?(gb|tb|mb|mm|cm|kg|gr|lts?|ml|w|hz|mah|ghz|pulgadas|")\b', case=False).astype(int)
    F['t_longest_tok'] = t.str.split().map(lambda w: max([len(x) for x in w], default=0))
    F['t_dup_words'] = t.str.split().map(lambda w: len(w) - len({x.lower() for x in w}))
    P = raw['pictures'].map(lambda v: v if isinstance(v, list) else [])
    def wh(p, key):
        m = re.match(r'(\d+)x(\d+)', p.get(key, '') or ''); return (int(m[1]), int(m[2])) if m else (np.nan, np.nan)
    first = P.map(lambda v: wh(v[0], 'size') if v else (np.nan, np.nan))
    firstmx = P.map(lambda v: wh(v[0], 'max_size') if v else (np.nan, np.nan))
    F['pic_w'] = first.map(lambda x: x[0]); F['pic_h'] = first.map(lambda x: x[1])
    F['pic_mw'] = firstmx.map(lambda x: x[0]); F['pic_mh'] = firstmx.map(lambda x: x[1])
    F['pic_ratio'] = F['pic_w'] / F['pic_h']; F['pic_max_mpx'] = F['pic_mw'] * F['pic_mh'] / 1e6
    F['pic_n_sizes'] = P.map(lambda v: len({p.get('size') for p in v}))
    F['pic_own_id'] = [np.mean([i in p.get('id', '') for p in v]) if v else np.nan for i, v in zip(raw['id'], P)]
    pdate = P.map(lambda v: re.search(r'_(\d{2})(\d{4})$', v[0].get('id', '')) if v else None)
    py = pdate.map(lambda m: int(m[2]) if m else np.nan); pm = pdate.map(lambda m: int(m[1]) if m else np.nan)
    dc = to_dt(raw['date_created'])
    F['pic_year'] = py; F['pic_gap_months'] = (dc.dt.year * 12 + dc.dt.month) - (py * 12 + pm)
    F['n_attr_vals_empty'] = raw['attributes'].map(lambda v: sum(1 for a in v if not a.get('value_name')) if isinstance(v, list) else 0)
    return F

# Palabras clave (minúsculas y SIN acentos, porque el texto ya se normalizó). Editá estas listas libremente.
USED_KW = ['estado','muy buen','lote','revista','antigua','antiguo','vinilo','lp','ano','made in',
           'usado','usada','usados','usadas','segunda mano','impecable','poco uso','detalles','para repuestos']
NEW_KW  = ['nuevo','nueva','nuevos','nuevas','kit','original','envio gratis','libro digital','sellado','sellada','sin abrir','0km','0 km','en caja']
WARR_USED = ['estado','buen','mi reputacion','mis calificaciones','calificaciones','descripcion','sin garantia']
WARR_NEW  = ['meses','ano','anos','de garantia','fabrica']
kw_re = lambda kws: re.compile(r'\b(' + '|'.join(re.escape(k) for k in kws) + r')\b')
RE_U, RE_N, RE_WU, RE_WN = kw_re(USED_KW), kw_re(NEW_KW), kw_re(WARR_USED), kw_re(WARR_NEW)
def kw_counts(S):
    return pd.DataFrame({'tx_kw_used': S['title'].map(lambda s: len(RE_U.findall(s))).values,
                         'tx_kw_new': S['title'].map(lambda s: len(RE_N.findall(s))).values,
                         'tx_kw_used_warr': S['warranty'].map(lambda s: len(RE_WU.findall(s))).values,
                         'tx_kw_new_warr': S['warranty'].map(lambda s: len(RE_WN.findall(s))).values})

def get_mats():
    if 'M' not in _MEM:
        Sa, Sb = get_sources()
        views = {
          'title_w':  (Sa['title'],    Sb['title'],    TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=80000, sublinear_tf=True)),
          'title_c':  (Sa['title'],    Sb['title'],    TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, max_features=120000, sublinear_tf=True)),
          'warranty': (Sa['warranty'], Sb['warranty'], TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
          'attrs':    (Sa['attr_vals'] + ' ' + Sa['attr_ids'], Sb['attr_vals'] + ' ' + Sb['attr_ids'], TfidfVectorizer(token_pattern=r'\S+', min_df=3, max_features=60000)),
          'vars':     (Sa['variations'], Sb['variations'], TfidfVectorizer(token_pattern=r'\S+', min_df=3)),
          'pics_meta':(Sa['pics'] + ' ' + Sa['meta'] + ' ' + Sa['payments'], Sb['pics'] + ' ' + Sb['meta'] + ' ' + Sb['payments'], TfidfVectorizer(token_pattern=r'\S+', min_df=3)),
        }
        mats = {n: (vec.fit_transform(a), vec.transform(b)) for n, (a, b, vec) in views.items()}
        _MEM['M'] = (mats, sparse.hstack([m[0] for m in mats.values()]).tocsr(), sparse.hstack([m[1] for m in mats.values()]).tocsr())
    return _MEM['M']
print('funciones de texto listas')

funciones de texto listas


## 6. Bloques de features (cada uno con su interruptor y su cache)
Cada bloque devuelve las columnas nuevas para train y test.

| Bloque | Qué hace | Interruptor |
|---|---|---|
| `text_basic` | estilo del título, fotos y palabras clave | siempre (es rápido) |
| `text_oof` | 8 modelos de texto; cada uno aporta la probabilidad de `used` calculada *fuera de fold* | `F_TEXT_OOF` |
| `knn_title` | etiqueta de los 10 títulos más parecidos del train (sin contar la propia fila) | `F_KNN_TITLE` |
| `time_neighbors` | etiqueta de las publicaciones del mismo vendedor cercanas en el tiempo (los vendedores suben en sesiones) | `F_TIME_NEIGHBORS` |
| `legacy_text` | 3 vistas de texto antiguas | `F_LEGACY_TEXT_VIEWS` |
| `siblings` | promedio de la predicción de "solo contenido" de las otras publicaciones del vendedor | `F_SIBLINGS` |
| `within_seller_title` | etiqueta del título más parecido *dentro* del mismo vendedor | `F_WITHIN_SELLER_TITLE` |
| `seller_dev` | qué tan distinta es la publicación de las otras del mismo vendedor (precio, fotos, garantía, tipo de listado). Sin usar el target | `F_SELLER_DEV` |
| `pair_te` | tasa de `used` de combinaciones como vendedor×tamaño de foto o categoría×garantía, suavizada hacia la tasa del vendedor/categoría | `F_PAIR_TE` |
| `text_cross` | modelos de texto donde cada palabra / tamaño de foto se **cruza con la categoría** (así el modelo aprende que "kit" o `500x375` significan algo distinto en cada categoría) | `F_TEXT_CROSS` |
| `seller_sim` | para cada vendedor busca los 20 más parecidos (categorías, vocabulario de títulos, tamaños de foto, garantía, precios) y usa su tasa de `used` como punto de partida; sirve sobre todo para vendedores sin publicaciones etiquetadas | `F_SELLER_SIM` |
| `cluster` | agrupa las publicaciones por tipo de producto (KMeans sobre los títulos, con train y test juntos) y calcula la tasa de `used` de cada grupo, de grupo×foto, grupo×garantía y, sobre todo, **vendedor×grupo** (un vendedor mixto suele vender usados de un tipo de producto y nuevos de otro) | `F_CLUSTER` |
| `svd_title` | descompone el TF-IDF de los títulos en 50 "temas" (SVD) y los usa como variables numéricas. Sin usar el target | `F_SVD_TITLE` |
| `text_nbsvm` | NB-SVM: cada palabra se pondera por cuánto favorece a `used` o `new` según Naive Bayes (calculado solo con los folds de entrenamiento) y sobre eso se entrena una regresión logística. Es un modelo de texto distinto a la regresión logística simple | `F_TEXT_NBSVM` |

Todos los que usan etiquetas lo hacen solo con *otros folds*, para no filtrar la respuesta.

In [6]:
def block_text_basic():
    Sa, Sb = get_sources()
    Fa = text_numeric_feats(train).add_prefix('tx_'); Fb = text_numeric_feats(test).add_prefix('tx_')
    ka, kb = kw_counts(Sa), kw_counts(Sb)
    for c in ka.columns: Fa[c] = ka[c].values; Fb[c] = kb[c].values
    return Fa, Fb

def block_text_oof():
    mats, Mtr_all, Mte_all = get_mats(); Sa, Sb = get_sources(); Fa, Fb = {}, {}
    def add(name, Mtr, Mte, model):
        Fa['tx_oof_' + name] = cross_val_predict(model, Mtr, yv, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]
        Fb['tx_oof_' + name] = model.fit(Mtr, yv).predict_proba(Mte)[:, 1]
        print(f'   {name:10s} acc OOF = {((Fa["tx_oof_" + name] > .5) == yv.values).mean():.4f}')
    for name, (Mtr, Mte) in mats.items(): add(name, Mtr, Mte, LogisticRegression(max_iter=3000, C=3))
    add('all', Mtr_all, Mte_all, LogisticRegression(max_iter=3000, C=1))
    cvb = CountVectorizer(ngram_range=(1, 2), min_df=2, binary=True)
    add('nb_title', cvb.fit_transform(Sa['title']), cvb.transform(Sb['title']), ComplementNB(alpha=0.5))
    return pd.DataFrame(Fa), pd.DataFrame(Fb)

def block_knn_title():
    from sklearn.neighbors import NearestNeighbors
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    Mtr = vec.fit_transform(df_o['title']); Mte = vec.transform(te_o['title']); K = 10
    nn = NearestNeighbors(n_neighbors=K + 1, metric='cosine', algorithm='brute', n_jobs=-1).fit(Mtr)
    def knn_feats(M, is_train):
        out = []
        for s in range(0, M.shape[0], 5000):
            d, ix = nn.kneighbors(M[s:s + 5000])
            if is_train: d = np.where(ix == np.arange(s, s + ix.shape[0])[:, None], np.inf, d)   # sacar la propia fila
            o = np.argsort(d, axis=1)[:, :K]
            d = np.take_along_axis(d, o, 1); ix = np.take_along_axis(ix, o, 1)
            sim = 1 - d; lab = yv.values[ix]; w = sim + 1e-6
            out.append(np.c_[(lab * w).sum(1) / w.sum(1), lab[:, :3].mean(1), lab[:, 0], sim[:, 0], sim.mean(1)])
        return np.vstack(out)
    names = ['knn_wmean', 'knn_top3', 'knn_top1', 'knn_sim1', 'knn_simmean']
    return pd.DataFrame(knn_feats(Mtr, True), columns=names), pd.DataFrame(knn_feats(Mte, False), columns=names)

def _fold_id():
    fid = np.zeros(ntr, int)
    for k, (_, va_i) in enumerate(skf.split(np.zeros(ntr), yv)): fid[va_i] = k
    return fid
def _groups():
    s_tr, s_te = train['seller_id'].values, test['seller_id'].values
    return (pd.Series(np.arange(ntr)).groupby(s_tr).apply(np.array), pd.Series(np.arange(nte)).groupby(s_te).apply(np.array))

def block_time_neighbors():
    ts_tr, ts_te = to_seconds(train['date_created']), to_seconds(test['date_created'])
    Y = yv.values.astype(float); fold_id = _fold_id(); groups_tr, groups_te = _groups()
    tcols = ['ti_n', 'ti_nn_lab', 'ti_nn_gap', 'ti_k3', 'ti_w60', 'ti_w3600', 'ti_prev_lab', 'ti_prev_gap',
             'ti_next_lab', 'ti_next_gap', 'ti_n60', 'ti_pos60', 'ti_n3600', 'ti_rate3600']
    def feats_time(tq, trf, valid, y_ref):
        q = len(tq); out = np.full((q, len(tcols)), np.nan); ar = np.arange(q)
        D = np.abs(tq[:, None] - trf[None, :]); D2 = np.where(valid, D, np.inf)
        n = valid.sum(1); has = n > 0; out[:, 0] = n
        j = D2.argmin(1); out[has, 1] = y_ref[j][has]; out[has, 2] = np.log1p(D2[ar, j])[has]
        o3 = np.argsort(D2, axis=1)[:, :3]; ok3 = np.isfinite(D2[ar[:, None], o3])
        out[has, 3] = ((y_ref[o3] * ok3).sum(1) / np.maximum(ok3.sum(1), 1))[has]
        for col, tau in [(4, 60.0), (5, 3600.0)]:
            w = np.where(valid, np.exp(-D / tau), 0.0); sw = w.sum(1)
            out[:, col] = np.where(sw > 1e-9, (w * y_ref).sum(1) / np.maximum(sw, 1e-12), np.nan)
        gap_prev = np.where(valid & (trf[None, :] <= tq[:, None]), tq[:, None] - trf[None, :], np.inf)
        gap_next = np.where(valid & (trf[None, :] >= tq[:, None]), trf[None, :] - tq[:, None], np.inf)
        for (a, b), G in [((6, 7), gap_prev), ((8, 9), gap_next)]:
            jj = G.argmin(1); ok = np.isfinite(G[ar, jj])
            out[ok, a] = y_ref[jj][ok]; out[ok, b] = np.log1p(G[ar, jj])[ok]
        m60 = valid & (D <= 60); m1h = valid & (D <= 3600)
        out[:, 10] = m60.sum(1); out[:, 11] = (m60 * y_ref).sum(1); out[:, 12] = m1h.sum(1)
        n1h = m1h.sum(1); out[n1h > 0, 13] = ((m1h * y_ref).sum(1) / np.maximum(n1h, 1))[n1h > 0]
        return out
    T_tr = np.full((ntr, len(tcols)), np.nan); T_te = np.full((nte, len(tcols)), np.nan)
    for s, idx in groups_tr.items():                    # train: referencias de OTROS folds del mismo vendedor
        T_tr[idx] = feats_time(ts_tr[idx], ts_tr[idx], fold_id[idx][:, None] != fold_id[idx][None, :], Y[idx])
    for s, idx in groups_te.items():                    # test: todas las publicaciones del vendedor en train
        ref = groups_tr.get(s)
        if ref is None: T_te[idx, 0] = 0; T_te[idx, 10] = 0; T_te[idx, 11] = 0; T_te[idx, 12] = 0; continue
        T_te[idx] = feats_time(ts_te[idx], ts_tr[ref], np.ones((len(idx), len(ref)), bool), Y[ref])
    return pd.DataFrame(T_tr, columns=tcols), pd.DataFrame(T_te, columns=tcols)

def block_legacy_text():
    def attr_text(series):
        return series.apply(lambda l: ' '.join(f"{a.get('id','')}_{str(a.get('value_name','')).replace(' ','_')}" for a in l) if isinstance(l, list) else '')
    views = {
      'title_word': (TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=30000, sublinear_tf=True), df_o['title'], te_o['title']),
      'title_char': (TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=5, max_features=60000, sublinear_tf=True), df_o['title'], te_o['title']),
      'attrs': (TfidfVectorizer(token_pattern=r'\S+', min_df=3, max_features=30000), attr_text(train['attributes']), attr_text(test['attributes'])),
    }
    Fa, Fb = {}, {}
    for name, (vec, ttr, tte) in views.items():
        Mtr = vec.fit_transform(ttr); Mte = vec.transform(tte)
        Fa['oof_' + name] = cross_val_predict(LogisticRegression(max_iter=2000, C=3), Mtr, yv, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]
        Fb['oof_' + name] = LogisticRegression(max_iter=2000, C=3).fit(Mtr, yv).predict_proba(Mte)[:, 1]
    return pd.DataFrame(Fa), pd.DataFrame(Fb)

def block_siblings():
    mats, Mtr_all, Mte_all = get_mats()
    seller_tr, seller_te = train['seller_id'].values, test['seller_id'].values; Y = yv.values
    p_tr = np.zeros(ntr); P_te_k = np.zeros((nte, 5)); fold_of_seller = {}
    for k, (tr_i, va_i) in enumerate(GroupKFold(5).split(Mtr_all, Y, groups=seller_tr)):      # cada fila se predice sin haber visto a su vendedor
        m = LogisticRegression(max_iter=3000, C=1).fit(Mtr_all[tr_i], Y[tr_i])
        p_tr[va_i] = m.predict_proba(Mtr_all[va_i])[:, 1]; P_te_k[:, k] = m.predict_proba(Mte_all)[:, 1]
        for s in np.unique(seller_tr[va_i]): fold_of_seller[s] = k
    kte = pd.Series(seller_te).map(fold_of_seller)
    p_te = np.where(kte.notna(), P_te_k[np.arange(nte), kte.fillna(0).astype(int).values], P_te_k.mean(1))
    allp = pd.DataFrame({'s': np.r_[seller_tr, seller_te], 'p': np.r_[p_tr, p_te]})
    pc = allp['p'].clip(1e-4, 1 - 1e-4); allp['l'] = np.log(pc / (1 - pc)); allp['hard'] = (allp['p'] > .5).astype(float)
    grp = allp.groupby('s'); n = grp['p'].transform('count').values
    def others(col):
        tot = grp[col].transform('sum').values
        return np.where(n > 1, (tot - allp[col].values) / np.maximum(n - 1, 1), np.nan)
    S = pd.DataFrame({'sib_n': n - 1, 'sib_p': others('p'), 'sib_logit': others('l'), 'sib_hard': others('hard')})
    return S.iloc[:ntr].reset_index(drop=True), S.iloc[ntr:].reset_index(drop=True)

def block_within_seller_title():
    mats, _, _ = get_mats(); Mtr_t, Mte_t = mats['title_w']
    lp_tr = np.log1p(train['price'].clip(lower=0).values); lp_te = np.log1p(test['price'].clip(lower=0).values)
    Y = yv.values.astype(float); fold_id = _fold_id(); groups_tr, groups_te = _groups()
    wcols = ['wi_n', 'wi_pos', 'wi_posrate', 'wi_top1_lab', 'wi_top1_sim', 'wi_wmean', 'wi_price_lab', 'wi_price_gap']
    def feats(S, valid, y_ref, dprice):
        q = S.shape[0]; out = np.full((q, len(wcols)), np.nan); ar = np.arange(q)
        n = valid.sum(1); has = n > 0
        S2 = np.where(valid, S, -np.inf); j = S2.argmax(1); D2 = np.where(valid, dprice, np.inf); jp = D2.argmin(1)
        w = np.where(valid, np.exp(10 * S), 0.0); pos = (valid * y_ref).sum(1)
        out[:, 0] = n; out[:, 1] = pos; out[has, 2] = (pos / np.maximum(n, 1))[has]
        out[has, 3] = y_ref[j][has]; out[has, 4] = S2[ar, j][has]
        out[has, 5] = ((w * y_ref).sum(1) / np.maximum(w.sum(1), 1e-12))[has]
        out[has, 6] = y_ref[jp][has]; out[has, 7] = D2[ar, jp][has]
        return out
    W_tr = np.full((ntr, len(wcols)), np.nan); W_te = np.full((nte, len(wcols)), np.nan)
    for s, idx in groups_tr.items():
        S = (Mtr_t[idx] @ Mtr_t[idx].T).toarray()
        W_tr[idx] = feats(S, fold_id[idx][:, None] != fold_id[idx][None, :], Y[idx], np.abs(lp_tr[idx][:, None] - lp_tr[idx][None, :]))
    for s, idx in groups_te.items():
        ref = groups_tr.get(s)
        if ref is None: W_te[idx, 0] = 0; W_te[idx, 1] = 0; continue
        S = (Mte_t[idx] @ Mtr_t[ref].T).toarray()
        W_te[idx] = feats(S, np.ones(S.shape, bool), Y[ref], np.abs(lp_te[idx][:, None] - lp_tr[ref][None, :]))
    return pd.DataFrame(W_tr, columns=wcols), pd.DataFrame(W_te, columns=wcols)

def _pic1(raw):
    return raw['pictures'].map(lambda l: (l[0].get('size') or 'none') if isinstance(l, list) and l else 'none')
def _warr_bucket(S):
    w = S['warranty']
    return np.select([w.str.contains('sin garantia').values, w.str.contains(r'\d+\s*(mes|ano)').values, (w == 'sin_dato').values],
                     ['sin', 'dur', 'none'], 'otro')

def block_seller_dev():
    Sa, Sb = get_sources()
    def table(d, raw, S):
        wh = np.array([(int(m[1]), int(m[2])) if m else (np.nan, np.nan)
                       for m in [re.match(r'(\d+)x(\d+)', sz) for sz in _pic1(raw).values]], dtype=float)
        return pd.DataFrame({'seller': d['seller_id'].astype(str).values,
            'price_log': d['price_log'].values, 'n_pictures': d['n_pictures'].values, 'title_len': d['title_len'].values,
            'n_attributes': d['n_attributes'].values, 'initial_quantity_log': d['initial_quantity_log'].values,
            'sold_quantity_log': d['sold_quantity_log'].values, 'pic_ratio': wh[:, 0] / wh[:, 1],
            'pic_size': _pic1(raw).values, 'warr': _warr_bucket(S), 'listing_type': d['listing_type_id'].astype(str).values,
            'ship_mode': d['shipping_mode'].astype(str).values, 'free_shipping': d['free_shipping'].astype(str).values,
            'category': d['category_id'].astype(str).values})
    comb = pd.concat([table(df_o, train, Sa), table(te_o, test, Sb)], ignore_index=True)
    sel = comb['seller']; out = pd.DataFrame(index=comb.index)
    for c in ['price_log', 'n_pictures', 'title_len', 'n_attributes', 'initial_quantity_log', 'sold_quantity_log', 'pic_ratio']:
        x = comb[c].astype(float); has = x.notna(); xs = x.where(has, 0.0)
        cnt = has.astype(int).groupby(sel).transform('sum'); sm = xs.groupby(sel).transform('sum')
        loo_n = (cnt - has.astype(int)).values
        loo_mean = np.where(loo_n > 0, (sm - xs).values / np.maximum(loo_n, 1), np.nan)      # media del vendedor SIN esta fila
        out['sd_dev_' + c] = np.where(has.values & (loo_n > 0), x.values - loo_mean, np.nan)
    n_sel = sel.map(sel.value_counts()).values
    for c in ['pic_size', 'warr', 'listing_type', 'ship_mode', 'free_shipping', 'category']:
        key = sel + '|' + comb[c]; n_pair = key.map(key.value_counts()).values
        out['sd_same_' + c] = np.where(n_sel > 1, (n_pair - 1) / np.maximum(n_sel - 1, 1), np.nan)   # % de las otras publicaciones del vendedor con el mismo valor
        out['sd_nuniq_' + c] = comb.groupby('seller')[c].transform('nunique').values
    return out.iloc[:ntr].reset_index(drop=True), out.iloc[ntr:].reset_index(drop=True)

def block_pair_te():
    Sa, Sb = get_sources()
    q = np.unique(np.quantile(Xa['price_rel_cat'], np.linspace(0, 1, 6)[1:-1]))           # 5 tramos de precio relativo a la categoría
    def table(d, raw, S, rel):
        return pd.DataFrame({'seller_id': d['seller_id'].astype(str).values, 'category_id': d['category_id'].astype(str).values,
                             'pic_size': _pic1(raw).values, 'warr': _warr_bucket(S), 'listing_type_id': d['listing_type_id'].astype(str).values,
                             'free_shipping': d['free_shipping'].astype(str).values, 'price_relbin': np.digitize(rel, q).astype(str)})
    Ta = table(df_o, train, Sa, Xa['price_rel_cat'].values); Tb = table(te_o, test, Sb, Xb['price_rel_cat'].values)
    prior = float(yv.mean()); yarr = yv.values.astype(float)
    def hier_te(a_tr, b_tr, a_te, b_te, sm_a=10.0, sm_ab=5.0):
        ab_tr = a_tr + '|' + b_tr; ab_te = a_te + '|' + b_te
        def fit(idx):          # estadísticas del padre (A) y del hijo (A,B), solo con las filas idx
            da = pd.DataFrame({'a': a_tr.values[idx], 'y': yarr[idx]}).groupby('a')['y'].agg(['sum', 'count'])
            pa = (da['sum'] + prior * sm_a) / (da['count'] + sm_a)
            dab = pd.DataFrame({'ab': ab_tr.values[idx], 'a': a_tr.values[idx], 'y': yarr[idx]}).groupby('ab').agg(s=('y', 'sum'), n=('y', 'count'), a=('a', 'first'))
            dab['enc'] = (dab['s'] + dab['a'].map(pa).values * sm_ab) / (dab['n'] + sm_ab)     # el hijo se suaviza hacia la tasa del padre
            return pa, dab
        def apply(pa, dab, a, ab):
            return ab.map(dab['enc']).fillna(a.map(pa).fillna(prior)).values, ab.map(dab['n']).fillna(0).values
        enc = np.zeros(ntr); cnt = np.zeros(ntr)
        for tr_i, va_i in skf.split(np.zeros(ntr), yv):
            pa, dab = fit(tr_i); enc[va_i], cnt[va_i] = apply(pa, dab, a_tr.iloc[va_i], ab_tr.iloc[va_i])
        pa, dab = fit(np.arange(ntr)); te_e, te_c = apply(pa, dab, a_te, ab_te)
        return enc, cnt, te_e, te_c
    short = {'seller_id': 'sel', 'category_id': 'cat', 'listing_type_id': 'lt'}
    PAIRS = [('seller_id', 'pic_size'), ('seller_id', 'price_relbin'), ('seller_id', 'warr'), ('seller_id', 'listing_type_id'),
             ('seller_id', 'free_shipping'), ('category_id', 'pic_size'), ('category_id', 'price_relbin'), ('category_id', 'warr'),
             ('category_id', 'listing_type_id'), ('category_id', 'free_shipping')]
    Fa, Fb = {}, {}
    for A, B in PAIRS:
        nm = f'{short.get(A, A)}_{short.get(B, B)}'
        e, c, te_e, te_c = hier_te(Ta[A], Ta[B], Tb[A], Tb[B])
        Fa['pte_' + nm], Fa['ptn_' + nm], Fb['pte_' + nm], Fb['ptn_' + nm] = e, c, te_e, te_c
    return pd.DataFrame(Fa), pd.DataFrame(Fb)

def block_text_cross():
    mats, Mtr_all, Mte_all = get_mats(); Sa, Sb = get_sources()
    def crossed(d, texts):
        cat = d['category_id'].astype(str).values
        return [' '.join(f'{c}__{t}' for t in re.findall(r'\S+', s)) for c, s in zip(cat, texts)]
    srcs = {'cat_title': (crossed(df_o, Sa['title']), crossed(te_o, Sb['title'])),          # categoría × palabra del título
            'cat_pics':  (crossed(df_o, Sa['pics']),  crossed(te_o, Sb['pics']))}           # categoría × tamaño de foto
    Fa, Fb = {}, {}; Xtr, Xte = [], []
    for name, (a, b) in srcs.items():
        vec = TfidfVectorizer(token_pattern=r'\S+', min_df=3, max_features=300000, sublinear_tf=True, lowercase=False)
        Mtr = vec.fit_transform(a); Mte = vec.transform(b); Xtr.append(Mtr); Xte.append(Mte)
        Fa['tx_oof_' + name] = cross_val_predict(LogisticRegression(max_iter=3000, C=3), Mtr, yv, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]
        Fb['tx_oof_' + name] = LogisticRegression(max_iter=3000, C=3).fit(Mtr, yv).predict_proba(Mte)[:, 1]
        print(f'   {name:10s} acc OOF = {((Fa["tx_oof_" + name] > .5) == yv.values).mean():.4f}')
    big_tr = sparse.hstack([Mtr_all] + Xtr).tocsr(); big_te = sparse.hstack([Mte_all] + Xte).tocsr()   # todo el texto + los cruces
    Fa['tx_oof_allx'] = cross_val_predict(LogisticRegression(max_iter=3000, C=1), big_tr, yv, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]
    Fb['tx_oof_allx'] = LogisticRegression(max_iter=3000, C=1).fit(big_tr, yv).predict_proba(big_te)[:, 1]
    print(f'   allx       acc OOF = {((Fa["tx_oof_allx"] > .5) == yv.values).mean():.4f}   (referencia: "all" sin cruces)')
    return pd.DataFrame(Fa), pd.DataFrame(Fb)

def block_seller_sim():
    from sklearn.decomposition import TruncatedSVD
    from sklearn.neighbors import NearestNeighbors
    Sa, Sb = get_sources()
    def table(d, raw, S):       # una fila por publicación (train + test), sin usar el target
        return pd.DataFrame({'seller': d['seller_id'].astype(str).values, 'price_log': d['price_log'].values, 'n_pictures': d['n_pictures'].values,
                             'title_len': d['title_len'].values, 'free_shipping': d['free_shipping'].values.astype(float),
                             'pic_size': _pic1(raw).values, 'warr': _warr_bucket(S), 'listing_type': d['listing_type_id'].astype(str).values,
                             'category': d['category_id'].astype(str).values})
    comb = pd.concat([table(df_o, train, Sa), table(te_o, test, Sb)], ignore_index=True)
    codes, uniq = pd.factorize(comb['seller']); n_sel = len(uniq); n_all = len(comb)
    # --- perfil de cada vendedor: estilo + categorías + vocabulario de sus títulos ---
    grp = comb.groupby(codes)
    num = pd.DataFrame({'n_log': np.log1p(grp.size().values), 'price_mean': grp['price_log'].mean().values, 'price_std': grp['price_log'].std().fillna(0).values,
                        'n_pic': grp['n_pictures'].mean().values, 'title_len': grp['title_len'].mean().values, 'free_ship': grp['free_shipping'].mean().values})
    num = ((num - num.mean()) / (num.std() + 1e-9)).clip(-4, 4).values
    def shares(col, top):
        v = comb[col].where(comb[col].isin(comb[col].value_counts().head(top).index), '_otro').reset_index(drop=True)
        return pd.crosstab(pd.Series(codes), v, normalize='index').values
    sty = np.hstack([shares('pic_size', 12), shares('warr', 4), shares('listing_type', 5)]); cat = shares('category', 40)
    titles = pd.concat([Sa['title'], Sb['title']], ignore_index=True)
    Mt = TfidfVectorizer(min_df=3, max_features=50000, sublinear_tf=True).fit_transform(titles)
    cnt = np.bincount(codes, minlength=n_sel)
    G = sparse.csr_matrix((1.0 / cnt[codes], (codes, np.arange(n_all))), shape=(n_sel, n_all))
    emb = TruncatedSVD(min(64, Mt.shape[1] - 1, n_sel - 1), random_state=RANDOM_STATE).fit_transform(G @ Mt)
    l2 = lambda A: A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)
    P = np.hstack([l2(emb), 0.7 * l2(num), 0.7 * l2(sty), l2(cat)])
    # --- los 20 vendedores más parecidos (sin contarse a sí mismo) ---
    K = 20
    d, ix = NearestNeighbors(n_neighbors=min(K + 1, n_sel), metric='cosine', algorithm='brute', n_jobs=-1).fit(P).kneighbors(P)
    d = np.where(ix == np.arange(n_sel)[:, None], np.inf, d)
    o = np.argsort(d, axis=1)[:, :K]; d = np.take_along_axis(d, o, 1); ix = np.take_along_axis(ix, o, 1)
    sim = np.clip(1 - d, 0, None); w = np.exp(10 * (sim - 1))
    W = sparse.csr_matrix((w.ravel(), (np.repeat(np.arange(n_sel), ix.shape[1]), ix.ravel())), shape=(n_sel, n_sel))
    wsum = np.maximum(np.asarray(W.sum(1)).ravel(), 1e-9)
    # --- tasa de 'used' de los parecidos, usando solo etiquetas de OTROS folds ---
    prior = float(yv.mean()); y = yv.values.astype(float); sel_tr, sel_te = codes[:ntr], codes[ntr:]
    def estimate(rows):
        S = np.bincount(sel_tr[rows], weights=y[rows], minlength=n_sel); C = np.bincount(sel_tr[rows], minlength=n_sel).astype(float)
        R = (S + 2 * prior) / (C + 2); Rel = C / (C + 3)                       # tasa suavizada de cada vendedor y su confiabilidad
        num_, den = W @ (Rel * R), W @ Rel
        return np.where(den > 1e-9, num_ / np.maximum(den, 1e-9), prior), den / wsum
    r_tr = np.zeros(ntr); q_tr = np.zeros(ntr)
    for tr_i, va_i in skf.split(np.zeros(ntr), yv):
        rate, rel = estimate(tr_i); r_tr[va_i] = rate[sel_tr[va_i]]; q_tr[va_i] = rel[sel_tr[va_i]]
    rate, rel = estimate(np.arange(ntr)); msim = sim.mean(1)
    return (pd.DataFrame({'ss_rate': r_tr, 'ss_rel': q_tr, 'ss_sim': msim[sel_tr]}),
            pd.DataFrame({'ss_rate': rate[sel_te], 'ss_rel': rel[sel_te], 'ss_sim': msim[sel_te]}))

def hier_te_global(a_tr, b_tr, a_te, b_te, sm_a=10.0, sm_ab=5.0):
    # tasa de 'used' del par (A,B), suavizada hacia la tasa de A; solo con etiquetas de OTROS folds. Devuelve (enc_train, cuenta_train, enc_test, cuenta_test)
    prior = float(yv.mean()); yarr = yv.values.astype(float)
    ab_tr = a_tr + '|' + b_tr; ab_te = a_te + '|' + b_te
    def fit(idx):
        da = pd.DataFrame({'a': a_tr.values[idx], 'y': yarr[idx]}).groupby('a')['y'].agg(['sum', 'count'])
        pa = (da['sum'] + prior * sm_a) / (da['count'] + sm_a)
        dab = pd.DataFrame({'ab': ab_tr.values[idx], 'a': a_tr.values[idx], 'y': yarr[idx]}).groupby('ab').agg(s=('y', 'sum'), n=('y', 'count'), a=('a', 'first'))
        dab['enc'] = (dab['s'] + dab['a'].map(pa).values * sm_ab) / (dab['n'] + sm_ab)
        return pa, dab
    def apply(pa, dab, a, ab):
        return ab.map(dab['enc']).fillna(a.map(pa).fillna(prior)).values, ab.map(dab['n']).fillna(0).values
    enc = np.zeros(ntr); cnt = np.zeros(ntr)
    for tr_i, va_i in skf.split(np.zeros(ntr), yv):
        pa, dab = fit(tr_i); enc[va_i], cnt[va_i] = apply(pa, dab, a_tr.iloc[va_i], ab_tr.iloc[va_i])
    pa, dab = fit(np.arange(ntr)); te_e, te_c = apply(pa, dab, a_te, ab_te)
    return enc, cnt, te_e, te_c

def block_cluster():
    from sklearn.decomposition import TruncatedSVD
    from sklearn.cluster import MiniBatchKMeans
    Sa, Sb = get_sources()
    titles = pd.concat([Sa['title'], Sb['title']], ignore_index=True)             # train + test juntos (no usa el target)
    Mt = TfidfVectorizer(min_df=3, max_features=60000, sublinear_tf=True).fit_transform(titles)
    emb = TruncatedSVD(min(100, Mt.shape[1] - 1), random_state=RANDOM_STATE).fit_transform(Mt)
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
    pic = pd.concat([_pic1(train), _pic1(test)], ignore_index=True).astype(str); warr = pd.Series(np.r_[_warr_bucket(Sa), _warr_bucket(Sb)]).astype(str)
    sel = pd.concat([train['seller_id'], test['seller_id']], ignore_index=True).astype(str)
    Fa, Fb = {}, {}
    for k in CLUSTER_KS:
        km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, batch_size=4096, n_init=3).fit(emb); lab = km.labels_
        cl = pd.Series(lab).astype(str); dist = np.linalg.norm(emb - km.cluster_centers_[lab], axis=1); size = np.bincount(lab, minlength=k)[lab]
        a_tr, a_te = cl.iloc[:ntr].reset_index(drop=True), cl.iloc[ntr:].reset_index(drop=True)
        Fa[f'cln_{k}'], Fb[f'cln_{k}'] = size[:ntr], size[ntr:]; Fa[f'cld_{k}'], Fb[f'cld_{k}'] = dist[:ntr], dist[ntr:]
        Fa[f'clte_{k}'], Fb[f'clte_{k}'] = target_encode(a_tr, a_te)                  # tasa de used del grupo
        for nm, (A, B) in {'pic': (cl, pic), 'warr': (cl, warr), 'sel': (sel, cl)}.items():   # grupo×foto, grupo×garantía, vendedor×grupo
            e, c, te_e, te_c = hier_te_global(A.iloc[:ntr].reset_index(drop=True), B.iloc[:ntr].reset_index(drop=True),
                                              A.iloc[ntr:].reset_index(drop=True), B.iloc[ntr:].reset_index(drop=True))
            Fa[f'clte_{k}_{nm}'], Fb[f'clte_{k}_{nm}'] = e, te_e
            if nm == 'sel': Fa[f'cltn_sel_{k}'], Fb[f'cltn_sel_{k}'] = c, te_c         # cuántas publicaciones etiquetadas tiene ese vendedor en ese grupo
    return pd.DataFrame(Fa), pd.DataFrame(Fb)

def block_svd_title():
    from sklearn.decomposition import TruncatedSVD
    Sa, Sb = get_sources()
    titles = pd.concat([Sa['title'], Sb['title']], ignore_index=True)             # train + test juntos (no usa el target)
    Mt = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=80000, sublinear_tf=True).fit_transform(titles)
    emb = TruncatedSVD(min(SVD_DIM, Mt.shape[1] - 1), random_state=RANDOM_STATE).fit_transform(Mt)
    cols = [f'sv_{i}' for i in range(emb.shape[1])]
    return pd.DataFrame(emb[:ntr], columns=cols), pd.DataFrame(emb[ntr:], columns=cols)

def block_text_nbsvm():
    from joblib import Parallel, delayed
    mats, Mtr_all, Mte_all = get_mats(); y = yv.values; Fa, Fb = {}, {}
    views = {'title_w': mats['title_w'], 'title_c': mats['title_c'], 'pics_meta': mats['pics_meta'], 'all': (Mtr_all, Mte_all)}
    def nb_ratio(M, rows):                                  # log-cociente de Naive Bayes, calculado SOLO con las filas de entrenamiento
        pos, neg = rows[y[rows] == 1], rows[y[rows] == 0]
        p = np.asarray(M[pos].sum(0)).ravel() + 1.0; q = np.asarray(M[neg].sum(0)).ravel() + 1.0
        return np.log((p / p.sum()) / (q / q.sum()))
    def fit_predict(Mtr, rows, Mpred):
        D = sparse.diags(nb_ratio(Mtr, rows))
        m = LogisticRegression(max_iter=2000, C=4).fit(Mtr[rows] @ D, y[rows])
        return m.predict_proba(Mpred @ D)[:, 1]
    for name, (Mtr, Mte) in views.items():
        folds = list(skf.split(np.zeros(ntr), yv))
        jobs = [delayed(fit_predict)(Mtr, tr_i, Mtr[va_i]) for tr_i, va_i in folds] + [delayed(fit_predict)(Mtr, np.arange(ntr), Mte)]
        res = Parallel(n_jobs=-1)(jobs); oof = np.zeros(ntr)
        for (tr_i, va_i), p in zip(folds, res[:-1]): oof[va_i] = p
        Fa['tx_oof_nbsvm_' + name], Fb['tx_oof_nbsvm_' + name] = oof, res[-1]
        print(f'   nbsvm {name:10s} acc OOF = {((oof > .5) == y).mean():.4f}')
    return pd.DataFrame(Fa), pd.DataFrame(Fb)
print('bloques definidos')

bloques definidos


### Ejecución de los bloques
Recorre la lista: si el interruptor está apagado, lo salta; si hay cache, lo carga; si no, lo calcula y lo guarda. Al terminar, `Xa` (train) y `Xb` (test) son las matrices finales de features.

In [7]:
KW_SIG = md5(USED_KW, NEW_KW, WARR_USED, WARR_NEW)      # si editás las palabras clave, text_basic se recalcula solo
BLOCKS = [('text_basic', True, block_text_basic, KW_SIG), ('text_oof', F_TEXT_OOF, block_text_oof, ''),
          ('knn_title', F_KNN_TITLE, block_knn_title, ''), ('time_neighbors', F_TIME_NEIGHBORS, block_time_neighbors, ''),
          ('legacy_text', F_LEGACY_TEXT_VIEWS, block_legacy_text, ''), ('siblings', F_SIBLINGS, block_siblings, ''),
          ('within_seller_title', F_WITHIN_SELLER_TITLE, block_within_seller_title, ''),
          ('seller_dev', F_SELLER_DEV, block_seller_dev, ''), ('pair_te', F_PAIR_TE, block_pair_te, ''),
          ('text_cross', F_TEXT_CROSS, block_text_cross, ''), ('seller_sim', F_SELLER_SIM, block_seller_sim, ''), ('cluster', F_CLUSTER, block_cluster, md5(CLUSTER_KS)),
          ('svd_title', F_SVD_TITLE, block_svd_title, md5(SVD_DIM)), ('text_nbsvm', F_TEXT_NBSVM, block_text_nbsvm, '')]
BLOCK_COLS = {}      # nombre del bloque -> columnas que aportó (se usa en la ablación)
for name, flag, fn, extra in BLOCKS:
    if not flag: print(f'  {name}: apagado'); continue
    key = f'blk_{name}_{md5(DATA_SIG, extra)[:20]}'
    got = cache_load(key)
    if got is None:
        t0 = time.time(); got = fn(); cache_save(key, got); print(f'  {name}: calculado en {time.time()-t0:.0f}s')
    else: print(f'  {name}: cargado del cache')
    Fa, Fb = got; BLOCK_COLS[name] = list(Fa.columns)
    Xa = pd.concat([Xa, Fa.reset_index(drop=True)], axis=1); Xb = pd.concat([Xb, Fb.reset_index(drop=True)], axis=1)
BERT_ALL = [d for d in ([BERT_DIR] + list(BERT_EXTRA_DIRS)) if d]      # corridas del notebook de BERT (mismos folds y mismo target)
if BERT_ALL and BERT_AS_FEATURE:
    for i_, d_ in enumerate(BERT_ALL):
        nm_ = 'bert' if i_ == 0 else f'bert{i_ + 1}'; a_ = pd.read_csv(f'{d_}/bert_feats_train.csv'); b_ = pd.read_csv(f'{d_}/bert_feats_test.csv')
        assert len(a_) == ntr and len(b_) == nte and (a_['y_check'].values == yv.values).all(), f'{d_}: el BERT no coincide con este train (¿SAMPLE_TRAIN? ¿otro target?)'
        Xa = pd.concat([Xa, a_[['bert_oof']].rename(columns={'bert_oof': nm_ + '_oof'}).reset_index(drop=True)], axis=1)
        Xb = pd.concat([Xb, b_[['bert_oof']].rename(columns={'bert_oof': nm_ + '_oof'}).reset_index(drop=True)], axis=1)
        BLOCK_COLS[nm_] = [nm_ + '_oof']; print(f'  {nm_}: agregado como feature ({nm_}_oof) desde {d_}')
Xa = Xa.copy(); Xb = Xb[Xa.columns].copy()
FEAT_SIG = md5(list(Xa.columns), pd.util.hash_pandas_object(Xa, index=False).values.tobytes(),
               pd.util.hash_pandas_object(Xb, index=False).values.tobytes())
print('\nMatriz final:', Xa.shape, '(train) |', Xb.shape, '(test)')

  text_basic: calculado en 11s
   title_w    acc OOF = 0.8193
   title_c    acc OOF = 0.8138
   warranty   acc OOF = 0.6267
   attrs      acc OOF = 0.5525
   vars       acc OOF = 0.5423
   pics_meta  acc OOF = 0.7306
   all        acc OOF = 0.8657
   nb_title   acc OOF = 0.8092
  text_oof: calculado en 114s
  knn_title: calculado en 169s
  time_neighbors: calculado en 8s
  legacy_text: apagado
  siblings: apagado
  within_seller_title: calculado en 22s
  seller_dev: calculado en 4s
  pair_te: calculado en 11s
  text_cross: apagado
  seller_sim: apagado
  cluster: apagado
  svd_title: apagado
  text_nbsvm: apagado
  bert: agregado como feature (bert_oof) desde /kaggle/input/notebooks/mirandacontieri/notebook-xlm-roberta-base
  bert2: agregado como feature (bert2_oof) desde /kaggle/input/notebooks/mirandacontieri/notebook-roberta-base-bne-v5-ap-2026
  bert3: agregado como feature (bert3_oof) desde /kaggle/input/notebooks/mirandacontieri/notebook-bert-titulos-ap

Matriz final: (70000, 203

## 6b. Evaluación rápida y ajuste de hiperparámetros (opcional)
- **`quick_cv`**: mide el accuracy OOF de LightGBM con un conjunto de columnas (learning rate alto, para comparar rápido).
- **`LGB_BASE`**: los hiperparámetros de LightGBM que usan los modelos de abajo (por defecto, los que venías usando).
- **`TUNE_LGB = True`**: busca mejores hiperparámetros con **Optuna** (optimiza el *logloss* de validación porque es menos ruidoso que el accuracy; poda las pruebas malas para ahorrar tiempo). Después compara "por defecto" contra "ajustado" **con semillas distintas** a las de la búsqueda, y guarda el resultado en `tuned_lgb.json`.
- **`USE_TUNED_PARAMS = True`**: usa ese archivo en los modelos finales. Conviene adoptarlo solo si la mejora supera ~0,001.

In [8]:
import json
LGB_FIXED = dict(n_estimators=5000, subsample_freq=1, verbose=-1, n_jobs=-1)
LGB_BASE_DEFAULT = dict(num_leaves=63, subsample=0.8, colsample_bytree=0.5, reg_lambda=1.0, min_child_samples=20)
LGB_BASE = dict(LGB_BASE_DEFAULT)
TUNED_PATH = os.path.join(CACHE_DIR, 'tuned_lgb.json')
acc = lambda p: ((p > .5) == yv.values).mean()

def quick_cv(cols, seeds=(1, 2), lr=0.08, base=None):
    import lightgbm as lgb
    base = LGB_BASE if base is None else base; oofs = []
    for seed in seeds:
        oof = np.zeros(ntr)
        for tr_i, va_i in skf.split(np.zeros(ntr), yv):
            m = lgb.LGBMClassifier(**{**LGB_FIXED, **base, 'learning_rate': lr, 'random_state': seed})
            m.fit(Xa.iloc[tr_i][cols], yv.iloc[tr_i], eval_set=[(Xa.iloc[va_i][cols], yv.iloc[va_i])], callbacks=[lgb.early_stopping(100, verbose=False)])
            oof[va_i] = m.predict_proba(Xa.iloc[va_i][cols])[:, 1]
        oofs.append(oof)
    return np.mean(oofs, axis=0)

if TUNE_LGB:
    try: import optuna
    except ImportError:
        import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna']); import optuna
    import lightgbm as lgb
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        p = dict(LGB_FIXED, n_estimators=3000, learning_rate=0.08, random_state=RANDOM_STATE,
                 num_leaves=trial.suggest_int('num_leaves', 15, 255, log=True),
                 max_depth=trial.suggest_categorical('max_depth', [-1, 6, 8, 10, 12]),
                 min_child_samples=trial.suggest_int('min_child_samples', 5, 100, log=True),
                 colsample_bytree=trial.suggest_float('colsample_bytree', 0.2, 0.9),
                 subsample=trial.suggest_float('subsample', 0.5, 1.0),
                 reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 30, log=True),
                 reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
                 min_split_gain=trial.suggest_float('min_split_gain', 0.0, 0.5))
        losses = []
        for k, (tr_i, va_i) in enumerate(skf.split(np.zeros(ntr), yv)):
            m = lgb.LGBMClassifier(**p)
            m.fit(Xa.iloc[tr_i], yv.iloc[tr_i], eval_set=[(Xa.iloc[va_i], yv.iloc[va_i])], eval_metric='binary_logloss',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            losses.append(m.best_score_['valid_0']['binary_logloss'])
            trial.report(float(np.mean(losses)), k)
            if trial.should_prune(): raise optuna.TrialPruned()
        return float(np.mean(losses))
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
    study.enqueue_trial(dict(LGB_BASE_DEFAULT, max_depth=-1, reg_alpha=1e-3, min_split_gain=0.0))   # empieza por tus valores actuales
    t0 = time.time(); study.optimize(objective, n_trials=N_TRIALS, timeout=TUNE_TIMEOUT_MIN * 60)
    done = [t for t in study.trials if t.state.name == 'COMPLETE']
    print(f'Optuna: {len(done)} pruebas completas de {len(study.trials)} en {time.time()-t0:.0f}s | mejor logloss: {study.best_value:.5f}')
    best = study.best_params; json.dump(best, open(TUNED_PATH, 'w')); print('mejores parámetros:', best)
    cols = list(Xa.columns)
    o_def = quick_cv(cols, seeds=(11, 12)); o_tun = quick_cv(cols, seeds=(11, 12), base={**LGB_BASE, **best})
    print(f'accuracy OOF (semillas nuevas) | por defecto: {acc(o_def):.4f} | ajustado: {acc(o_tun):.4f} | diferencia: {acc(o_tun)-acc(o_def):+.4f}')
    print('Adoptalo (USE_TUNED_PARAMS = True) solo si la diferencia supera ~ +0.001.')

if USE_TUNED_PARAMS:
    if os.path.exists(TUNED_PATH): LGB_BASE.update(json.load(open(TUNED_PATH))); print('LightGBM usará parámetros ajustados:', LGB_BASE)
    else: print('USE_TUNED_PARAMS=True pero no existe', TUNED_PATH, '-> se usan los parámetros por defecto')

## 7. Modelos
Los tres usan los mismos 5 folds. Cada modelo genera predicciones **OOF** (para train, evaluadas fuera de fold) y promedia sus 5 modelos de fold para el test. Las predicciones se guardan en cache con una firma de las features y los parámetros: si no cambiaste nada, no se reentrena.

- **LightGBM:** promedia varias semillas (`SEEDS`).
- **XGBoost:** con los nombres de columnas limpios (no acepta `[ ] < >`).
- **CatBoost:** recibe vendedor, categoría y ciudad como categorías propias y **no** ve las features que ya resumen la etiqueta del vendedor (`*_te`, `knn_`, `sib_`, `wi_`, `ti_`, `pte_`, `ptn_`, `ss_rate`, `ss_rel`), para que sea distinto de los otros dos.

In [9]:
preds = {}      # nombre -> (predicción OOF en train, predicción en test)
acc = lambda p: ((p > .5) == yv.values).mean()
def run_cached(name, fn, *sig):
    key = f'pred_{name}_{FEAT_SIG[:12]}_{md5(*sig)[:8]}'
    got = cache_load(key)
    if got is None:
        t0 = time.time(); got = fn(); cache_save(key, got); print(f'  {name}: entrenado en {time.time()-t0:.0f}s')
    else: print(f'  {name}: cargado del cache')
    return got

# ---------------- LightGBM ----------------
if RUN_LGB:
    import lightgbm as lgb
    def lgb_cv(seed):
        params = {**LGB_FIXED, **LGB_BASE, 'learning_rate': LGB_LR, 'random_state': seed}
        oof = np.zeros(ntr); tp = np.zeros(nte)
        for tr_i, va_i in skf.split(np.zeros(ntr), yv):
            m = lgb.LGBMClassifier(**params)
            m.fit(Xa.iloc[tr_i], yv.iloc[tr_i], eval_set=[(Xa.iloc[va_i], yv.iloc[va_i])], callbacks=[lgb.early_stopping(100, verbose=False)])
            oof[va_i] = m.predict_proba(Xa.iloc[va_i])[:, 1]; tp += m.predict_proba(Xb)[:, 1] / N_FOLDS
        return oof, tp
    sig_extra = () if LGB_BASE == LGB_BASE_DEFAULT else (sorted(LGB_BASE.items()),)      # el cache viejo sigue valiendo si no cambiaste parámetros
    runs = [run_cached(f'lgb_s{s}', lambda s=s: lgb_cv(s), LGB_LR, s, *sig_extra) for s in SEEDS]
    preds['lgb'] = (np.mean([r[0] for r in runs], axis=0), np.mean([r[1] for r in runs], axis=0))
    print(f'  LightGBM ({len(SEEDS)} semilla/s): CV acc = {acc(preds["lgb"][0]):.4f}')

# ---------------- XGBoost ----------------
if RUN_XGB:
    import xgboost as xgb
    clean_cols = lambda d: d.set_axis([re.sub(r'[\[\]<>]', '_', str(c)) for c in d.columns], axis=1)
    def xgb_cv(seed=RANDOM_STATE):
        Xa_x, Xb_x = clean_cols(Xa), clean_cols(Xb); oof = np.zeros(ntr); tp = np.zeros(nte)
        for tr_i, va_i in skf.split(np.zeros(ntr), yv):
            m = xgb.XGBClassifier(n_estimators=5000, learning_rate=LGB_LR, max_depth=7, min_child_weight=2, subsample=0.8, colsample_bytree=0.5,
                                  reg_lambda=1.0, tree_method='hist', eval_metric='logloss', early_stopping_rounds=100, n_jobs=-1, random_state=seed)
            m.fit(Xa_x.iloc[tr_i], yv.iloc[tr_i], eval_set=[(Xa_x.iloc[va_i], yv.iloc[va_i])], verbose=False)
            oof[va_i] = m.predict_proba(Xa_x.iloc[va_i])[:, 1]; tp += m.predict_proba(Xb_x)[:, 1] / N_FOLDS
        return oof, tp
    seeds_x = ([RANDOM_STATE] + [7, 11, 13, 17])[:1 if FAST_MODE else max(1, min(N_SEEDS_XGB, 5))]      # la primera semilla conserva el cache anterior
    runs_x = [run_cached('xgb' if s == RANDOM_STATE else f'xgb_s{s}', lambda s=s: xgb_cv(s), LGB_LR) for s in seeds_x]
    preds['xgb'] = (np.mean([r[0] for r in runs_x], axis=0), np.mean([r[1] for r in runs_x], axis=0))
    print(f'  XGBoost: CV acc = {acc(preds["xgb"][0]):.4f}')

# ---------------- CatBoost ----------------
if RUN_CAT_EFF:
    from catboost import CatBoostClassifier
    is_label_feat = lambda c: c.endswith('_te') or c.startswith(('knn_', 'sib_', 'wi_', 'ti_', 'pte_', 'ptn_', 'ss_rate', 'ss_rel', 'clte_', 'emb_', 'nn_'))
    num_cols = [c for c in Xa.columns if not is_label_feat(c)]
    cat_frame = lambda raw, o: pd.DataFrame({'seller_id': raw['seller_id'].astype(str).values, 'category_id': raw['category_id'].astype(str).values,
                                             'city': o['city'].astype(str).values, 'seller_cat': o['seller_cat'].astype(str).values})
    def cat_cv(seed=RANDOM_STATE):
        Xc_tr = pd.concat([Xa[num_cols].reset_index(drop=True), cat_frame(train, df_o)], axis=1)
        Xc_te = pd.concat([Xb[num_cols].reset_index(drop=True), cat_frame(test, te_o)], axis=1)
        cat_idx = list(range(len(num_cols), Xc_tr.shape[1])); Y = yv.values; oof = np.zeros(ntr); tp = np.zeros(nte)
        for k, (tr_i, va_i) in enumerate(skf.split(np.zeros(ntr), yv)):
            m = CatBoostClassifier(iterations=2000, learning_rate=0.08, depth=8, l2_leaf_reg=3, eval_metric=CAT_EVAL_METRIC,
                                   early_stopping_rounds=150, thread_count=-1, random_seed=seed, verbose=0)
            m.fit(Xc_tr.iloc[tr_i], Y[tr_i], cat_features=cat_idx, eval_set=(Xc_tr.iloc[va_i], Y[va_i]))
            oof[va_i] = m.predict_proba(Xc_tr.iloc[va_i])[:, 1]; tp += m.predict_proba(Xc_te)[:, 1] / N_FOLDS
            print(f'     CatBoost fold {k}: acc = {((oof[va_i] > .5) == Y[va_i]).mean():.4f}')
        return oof, tp
    seeds_c = ([RANDOM_STATE] + [7, 11, 13, 17])[:max(1, min(N_SEEDS_CAT, 5))]
    sig_c = ('cat_v1', *(() if CAT_EVAL_METRIC == 'Accuracy' else (CAT_EVAL_METRIC,)))                  # con 'Accuracy' el cache viejo sigue valiendo
    runs_c = [run_cached('cat' if s == RANDOM_STATE else f'cat_s{s}', lambda s=s: cat_cv(s), *sig_c) for s in seeds_c]
    preds['cat'] = (np.mean([r[0] for r in runs_c], axis=0), np.mean([r[1] for r in runs_c], axis=0))
    print(f'  CatBoost: CV acc = {acc(preds["cat"][0]):.4f}')

  lgb_s1: entrenado en 91s
  lgb_s2: entrenado en 86s
  lgb_s3: entrenado en 89s
  lgb_s4: entrenado en 81s
  lgb_s5: entrenado en 83s
  lgb_s6: entrenado en 83s
  LightGBM (6 semilla/s): CV acc = 0.9459
  xgb: entrenado en 320s
  xgb_s7: entrenado en 331s
  xgb_s11: entrenado en 309s
  XGBoost: CV acc = 0.9458
     CatBoost fold 0: acc = 0.9461
     CatBoost fold 1: acc = 0.9419
     CatBoost fold 2: acc = 0.9429
     CatBoost fold 3: acc = 0.9461
     CatBoost fold 4: acc = 0.9455
  cat: entrenado en 388s
     CatBoost fold 0: acc = 0.9441
     CatBoost fold 1: acc = 0.9411
     CatBoost fold 2: acc = 0.9454
     CatBoost fold 3: acc = 0.9459
     CatBoost fold 4: acc = 0.9454
  cat_s7: entrenado en 352s
     CatBoost fold 0: acc = 0.9450
     CatBoost fold 1: acc = 0.9414
     CatBoost fold 2: acc = 0.9457
     CatBoost fold 3: acc = 0.9464
     CatBoost fold 4: acc = 0.9459
  cat_s11: entrenado en 358s
  CatBoost: CV acc = 0.9453


## 8. Mezcla de modelos y submission
Combina las predicciones de los modelos activos:
- `BLEND = 'mean'`: promedio simple.
- `BLEND = 'meta'`: una regresión logística aprende cuánto pesar a cada modelo (se evalúa con folds distintos para no engañarse).

Si `NN_DIR` apunta a la carpeta de la red neuronal, se suma como un modelo más. El archivo final es `submission.csv`, con `ID` desde 1.

In [10]:
if NN_DIR:      # opcional: red neuronal entrenada en otro notebook (mismos folds y mismo target)
    a = pd.read_csv(f'{NN_DIR}/nn_feats_train.csv'); b = pd.read_csv(f'{NN_DIR}/nn_feats_test.csv')
    assert len(a) == ntr and len(b) == nte and (a['y_check'].values == yv.values).all(), 'la red no coincide con este train (¿SAMPLE_TRAIN?)'
    preds['nn'] = (a['nn_oof'].values, b['nn_oof'].values); print(f'  red neuronal: CV acc = {acc(preds["nn"][0]):.4f}')

if BERT_IN_BLEND:      # opcional: cada BERT también como modelo de la mezcla
    for i_, d_ in enumerate([d for d in ([BERT_DIR] + list(BERT_EXTRA_DIRS)) if d]):
        nm_ = 'bert' if i_ == 0 else f'bert{i_ + 1}'; a = pd.read_csv(f'{d_}/bert_feats_train.csv'); b = pd.read_csv(f'{d_}/bert_feats_test.csv')
        assert len(a) == ntr and len(b) == nte and (a['y_check'].values == yv.values).all(), f'{d_}: el BERT no coincide con este train'
        preds[nm_] = (a['bert_oof'].values, b['bert_oof'].values); print(f'  {nm_}: CV acc = {acc(preds[nm_][0]):.4f}')

assert preds, 'no hay ningún modelo activo (RUN_LGB / RUN_XGB / RUN_CAT)'
names = list(preds); O = np.c_[[preds[n][0] for n in names]].T; T = np.c_[[preds[n][1] for n in names]].T
lg = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))
print('\nCV accuracy por modelo:', {n: round(acc(preds[n][0]), 4) for n in names})
if len(names) > 1: print('\nCorrelación entre modelos (OOF):'); print(pd.DataFrame(O, columns=names).corr().round(3))
print('promedio simple:', round(acc(O.mean(1)), 4))

if BLEND == 'meta' and len(names) > 1:
    meta = LogisticRegression(C=1.0)
    o_final = cross_val_predict(meta, lg(O), yv, cv=StratifiedKFold(N_FOLDS, shuffle=True, random_state=123), method='predict_proba')[:, 1]
    meta.fit(lg(O), yv); t_final = meta.predict_proba(lg(T))[:, 1]
    print('meta-modelo (CV):', round(acc(o_final), 4), '| pesos:', dict(zip(names, meta.coef_[0].round(2))))
else:
    o_final, t_final = O.mean(1), T.mean(1)
print(f'\n>>> CV accuracy de la mezcla final ({BLEND}): {acc(o_final):.4f}')

sub = pd.DataFrame({'ID': range(1, nte + 1), TARGET: [classes[i] for i in (t_final > .5).astype(int)]})
sub.to_csv('submission.csv', index=False)
if SAVE_COPY: sub.to_csv(f'submission_cv{acc(o_final):.4f}_{time.strftime("%H%M")}.csv', index=False)
print(sub[TARGET].value_counts(normalize=True).round(3).to_dict(), '| guardado: submission.csv')


CV accuracy por modelo: {'lgb': np.float64(0.9459), 'xgb': np.float64(0.9458), 'cat': np.float64(0.9453)}

Correlación entre modelos (OOF):
       lgb    xgb    cat
lgb  1.000  0.999  0.996
xgb  0.999  1.000  0.996
cat  0.996  0.996  1.000
promedio simple: 0.9463
meta-modelo (CV): 0.9462 | pesos: {'lgb': np.float64(0.34), 'xgb': np.float64(0.38), 'cat': np.float64(0.25)}

>>> CV accuracy de la mezcla final (meta): 0.9462
{'new': 0.528, 'used': 0.472} | guardado: submission.csv


## 9. Diagnóstico opcional (`SHOW_DIAGNOSTICS = True`)
Accuracy de la mezcla según el vendedor tenga otras publicaciones en el train (*repetido*) o sea único. Sirve para ver dónde están los errores.

In [11]:
if SHOW_DIAGNOSTICS:
    seen = train['seller_id'].map(train['seller_id'].value_counts()).values > 1
    ok = ((o_final > .5) == yv.values)
    print(f'vendedor repetido: {ok[seen].mean():.4f} (n={seen.sum()}) | vendedor único: {ok[~seen].mean():.4f} (n={(~seen).sum()})')
    print(f'test con vendedor visto en train: {test["seller_id"].isin(set(train["seller_id"])).mean():.3f}')
else:
    print('diagnóstico apagado')

vendedor repetido: 0.9657 (n=50074) | vendedor único: 0.8973 (n=19926)
test con vendedor visto en train: 0.716


## 10. Ablación de bloques (opcional, `RUN_ABLATION = True`)
Para cada bloque de features activo, reentrena LightGBM **sin ese bloque** y compara contra el modelo con todo. Una diferencia negativa grande significa que el bloque ayuda; si queda cerca de 0 (menos de ~0,0003, que es el ruido entre semillas), el bloque no aporta y podés apagarlo para ahorrar tiempo.\n\n**Ojo:** si dos bloques aportan la misma información, quitar uno solo no se nota (el otro lo compensa). Por eso también se miden **grupos** (`ABL_GROUPS`) quitando varios bloques a la vez.

In [12]:
if RUN_ABLATION:
    all_cols = list(Xa.columns); t0 = time.time()
    base_o = quick_cv(all_cols, seeds=ABL_SEEDS); base_acc = acc(base_o)
    print(f'todas las features ({len(all_cols)}): {base_acc:.4f}')
    for name, cols_b in ([] if ABL_SKIP_SINGLE else BLOCK_COLS.items()):
        drop = set(cols_b); o = quick_cv([c for c in all_cols if c not in drop], seeds=ABL_SEEDS)
        print(f'  sin {name:20s} ({len(cols_b):2d} cols): {acc(o):.4f}   diferencia {acc(o) - base_acc:+.4f}')
    for gname, blocks in ABL_GROUPS.items():          # grupos: bloques parecidos se quitan JUNTOS (si uno compensa al otro, solo así se nota)
        bl = [b for b in blocks if b in BLOCK_COLS]
        if len(bl) < 2: continue
        drop = set(c for b in bl for c in BLOCK_COLS[b]); o = quick_cv([c for c in all_cols if c not in drop], seeds=ABL_SEEDS)
        print(f'  sin GRUPO {gname:42s} ({len(drop):2d} cols): {acc(o):.4f}   diferencia {acc(o) - base_acc:+.4f}')
    print(f'ablación terminada en {time.time()-t0:.0f}s')
else:
    print('ablación apagada')

todas las features (203): 0.9454
  sin GRUPO vecinos (knn_title + time_neighbors)       (19 cols): 0.9454   diferencia +0.0000
  sin GRUPO vendedor nuevos (seller_dev + pair_te)     (39 cols): 0.9441   diferencia -0.0013
  sin GRUPO todos los BERT                             ( 3 cols): 0.9437   diferencia -0.0017
  sin GRUPO todos vendedor/vecinos                     (58 cols): 0.9440   diferencia -0.0014
ablación terminada en 346s


## 11. Laboratorio de iteraciones (`RUN_LAB = True`)
Sirve para ir mejorando de a poco **sin sobreajustar el público**. Todo se decide con el **OOF del train** (miles de filas con la respuesta conocida), nunca con el puntaje público.

**Qué hace**
1. **¿Dónde están los errores?** Tabla de errores según la confianza del modelo y por segmento (vendedor, garantía, categoría). Te dice cuánto se puede ganar *de verdad* mirando las filas dudosas y cuántos errores son "seguros" (probablemente ruido de etiquetas).
2. **Candidatos:** variantes de la mezcla y un *modelo especialista* entrenado solo con las filas dudosas.
3. **Veredicto por candidato:** compara contra la mezcla actual (o contra una corrida anterior si ponés `REF_RUN`) con el CV, con una prueba de significancia (filas que mejoran vs. empeoran) y cuenta **cuántas filas del test cambian**. Solo los que cumplen `LAB_MIN_DELTA` y `LAB_MIN_Z` quedan en `laboratorio/para_enviar/`.
4. **Registro:** cada corrida se anota en `laboratorio/registro_experimentos.csv`. Cuando conozcas el puntaje público lo anotás con `registrar_publico('nombre', 0.9441)`.

**Reglas para no sobreajustar el público**
- El **CV decide**; el público solo sirve de chequeo de cordura (¿bajó muchísimo? entonces hay un error de código).
- Un cambio que mueve pocas filas del test **no se puede medir en el público**: el ruido de la diferencia entre dos envíos es ~ ±2·√(filas que cambian)/30000. La tabla lo calcula.
- No elijas entre muchas variantes por el puntaje público: con muchas variantes, la "mejor" del público lo es en parte por azar.
- No hace falta usar los 5 envíos diarios.

In [13]:
LOG = os.path.join(LAB_DIR, 'registro_experimentos.csv')
def registrar_publico(nombre, puntaje, notas=''):
    # anota el puntaje público en el registro (busca el último candidato cuyo nombre contenga `nombre`)
    log = pd.read_csv(LOG); log['notas'] = log['notas'].astype(object); idx = log.index[log['candidato'].str.contains(nombre, regex=False)]
    if len(idx) == 0: print('no encontré ese candidato en', LOG); return
    log.loc[idx[-1], 'publico'] = float(puntaje); log.loc[idx[-1], 'notas'] = notas
    log.to_csv(LOG, index=False); print('registrado:', log.loc[idx[-1], 'candidato'], '->', puntaje)

if RUN_LAB:
    import lightgbm as lgb
    os.makedirs(os.path.join(LAB_DIR, 'para_enviar'), exist_ok=True)
    y_ = yv.values; N_ = len(y_); p_oof, p_te = o_final, t_final; ok_base = ((p_oof > .5) == y_)
    stamp = time.strftime('%Y%m%d_%H%M')
    np.save(os.path.join(LAB_DIR, f'{stamp}_base_oof.npy'), p_oof); np.save(os.path.join(LAB_DIR, f'{stamp}_base_te.npy'), p_te)   # para comparar corridas futuras (REF_RUN)
    print(f'corrida guardada como {stamp}: para compararla en el futuro, poné REF_RUN = "{stamp}"\n')
    err_total = int((~ok_base).sum())

    # ---------- A) ¿dónde están los errores? (solo OOF del train) ----------
    cut = pd.cut(p_oof, [0, .02, .05, .15, .35, .65, .85, .95, .98, 1.0], include_lowest=True)
    tab = pd.DataFrame({'bin': cut, 'ok': ok_base}).groupby('bin', observed=True)['ok'].agg(filas='size', accuracy='mean')
    tab['errores'] = (tab['filas'] * (1 - tab['accuracy'])).round().astype(int); tab['% de errores'] = (100 * tab['errores'] / max(tab['errores'].sum(), 1)).round(1)
    print('Errores según la probabilidad de la mezcla actual (OOF):'); print(tab.round(4).to_string())
    lo, hi = UNC_BAND; U = (p_oof > lo) & (p_oof < hi); Ut = (p_te > lo) & (p_te < hi); err_U = int((~ok_base[U]).sum())
    print(f'\nBanda dudosa {UNC_BAND}: {U.mean():.1%} de las filas del train ({Ut.mean():.1%} del test) concentran {err_U / max(err_total, 1):.1%} de los errores.')
    print(f'Techo si acertaras TODA la banda dudosa: {1 - (err_total - err_U) / N_:.4f} | errores "seguros" fuera de la banda: {err_total - err_U} '
          f'({(err_total - err_U) / max(err_total, 1):.1%}), difíciles de arreglar: suelen ser ruido de etiquetas.')
    Sa_, _ = get_sources()
    seg = pd.DataFrame({'ok': ok_base, 'categoria': train['category_id'].values, 'garantia': _warr_bucket(Sa_),
                        'vendedor': np.where(train['seller_id'].map(train['seller_id'].value_counts()).values > 1, 'repetido', 'unico')})
    def seg_tab(col, top=None):
        t = seg.groupby(col)['ok'].agg(filas='size', accuracy='mean'); t['errores'] = (t['filas'] * (1 - t['accuracy'])).round().astype(int)
        t = t.sort_values('errores', ascending=False); return t.head(top) if top else t
    for col, top in [('vendedor', None), ('garantia', None), ('categoria', 8)]:
        print(f'\nErrores por {col}:'); print(seg_tab(col, top).round(4).to_string())

    # ---------- B) candidatos ----------
    cands = {'base_actual': (p_oof, p_te, f'mezcla {BLEND} actual (referencia)')}
    names_ = list(preds); O_ = np.c_[[preds[n][0] for n in names_]].T; T_ = np.c_[[preds[n][1] for n in names_]].T
    cands['promedio_simple'] = (O_.mean(1), T_.mean(1), 'promedio simple de los modelos')
    if 'lgb' in preds and 'xgb' in preds:
        cands['lgb_xgb'] = (np.mean([preds['lgb'][0], preds['xgb'][0]], axis=0), np.mean([preds['lgb'][1], preds['xgb'][1]], axis=0), 'solo LightGBM + XGBoost')

    if U.sum() >= 500:      # modelo especialista: se entrena SOLO con las filas dudosas (usa las features + la probabilidad de la mezcla)
        Xs_a = Xa.copy(); Xs_b = Xb.copy(); Xs_a['stage1_logit'] = lg(p_oof); Xs_b['stage1_logit'] = lg(p_te)
        sp = {**LGB_FIXED, 'n_estimators': 2000, 'learning_rate': 0.03, 'num_leaves': 15, 'min_child_samples': 30,
              'colsample_bytree': 0.5, 'subsample': 0.8, 'reg_lambda': 5.0, 'random_state': 7}
        o_sp = np.full(N_, np.nan); t_sp = np.zeros(nte)
        for tr_i, va_i in skf.split(np.zeros(N_), yv):
            tr_u, va_u = tr_i[U[tr_i]], va_i[U[va_i]]
            m = lgb.LGBMClassifier(**sp).fit(Xs_a.iloc[tr_u], yv.iloc[tr_u], eval_set=[(Xs_a.iloc[va_u], yv.iloc[va_u])],
                                             callbacks=[lgb.early_stopping(100, verbose=False)])
            o_sp[va_u] = m.predict_proba(Xs_a.iloc[va_u])[:, 1]; t_sp[Ut] += m.predict_proba(Xs_b[Ut])[:, 1] / N_FOLDS
        accw = {}
        for w in (0.25, 0.5, 0.75, 1.0):      # cuánto peso darle al especialista dentro de la banda dudosa
            po = p_oof.copy(); po[U] = (1 - w) * p_oof[U] + w * o_sp[U]; accw[w] = acc(po)
        wb = max(accw, key=accw.get); po = p_oof.copy(); po[U] = (1 - wb) * p_oof[U] + wb * o_sp[U]
        pt = p_te.copy(); pt[Ut] = (1 - wb) * p_te[Ut] + wb * t_sp[Ut]
        cands[f'especialista_w{wb}'] = (po, pt, f'especialista entrenado solo en la banda dudosa {UNC_BAND}, peso {wb} (acc OOF por peso: ' +
                                       ', '.join(f'{k}: {v:.4f}' for k, v in accw.items()) + ')')
    else: print('\nla banda dudosa tiene muy pocas filas: se omite el especialista')

    # ---------- C) veredicto de cada candidato ----------
    ref = None
    if REF_SUBMISSION and os.path.exists(REF_SUBMISSION):
        r = pd.read_csv(REF_SUBMISSION)
        if len(r) == nte: ref = (r[TARGET] == classes[-1]).astype(int).values
    ref_loaded = False
    if REF_RUN:
        f1, f2 = os.path.join(LAB_DIR, f'{REF_RUN}_base_oof.npy'), os.path.join(LAB_DIR, f'{REF_RUN}_base_te.npy')
        if os.path.exists(f1) and os.path.exists(f2):
            cmp_oof, cmp_te = np.load(f1), np.load(f2); ref_loaded = len(cmp_oof) == N_ and len(cmp_te) == nte
        if not ref_loaded: print(f'⚠ no pude usar REF_RUN={REF_RUN} (¿falta el archivo o cambió SAMPLE_TRAIN?): se compara contra la mezcla actual')
    if not ref_loaded: cmp_oof, cmp_te = p_oof, p_te
    ok_cmp = ((cmp_oof > .5) == y_); base_te = (cmp_te > .5).astype(int); rows = []
    print(f'Comparación contra: {"la corrida " + REF_RUN if ref_loaded else "la mezcla actual"} (CV de referencia: {ok_cmp.mean():.5f})')
    for name, (po, pt, desc) in cands.items():
        ok = ((po > .5) == y_); gg = int((ok & ~ok_cmp).sum()); ll = int((~ok & ok_cmp).sum())
        delta = ok.mean() - ok_cmp.mean(); z = (gg - ll) / np.sqrt(gg + ll) if gg + ll > 0 else 0.0
        pred = (pt > .5).astype(int); n_base = int((pred != base_te).sum()); n_ref = int((pred != ref).sum()) if ref is not None else None
        d_pub = n_ref if n_ref is not None else n_base; res = np.nan if (name == 'base_actual' and not ref_loaded) else 2 * np.sqrt(max(d_pub, 1)) / nte
        if name == 'base_actual' and not ref_loaded: veredicto = 'referencia'
        elif delta < LAB_MIN_DELTA: veredicto = f'NO: mejora de CV < {LAB_MIN_DELTA}'
        elif z < LAB_MIN_Z: veredicto = f'NO: z = {z:.1f} < {LAB_MIN_Z} (podría ser azar)'
        else: veredicto = 'ENVIAR'
        cid = f'{stamp}_{name}'
        if veredicto == 'ENVIAR':
            pd.DataFrame({'ID': range(1, nte + 1), TARGET: [classes[i] for i in pred]}).to_csv(os.path.join(LAB_DIR, 'para_enviar', f'{cid}_cv{acc(po):.4f}.csv'), index=False)
        rows.append({'candidato': cid, 'descripcion': desc, 'cv': round(acc(po), 5), 'delta_cv': round(delta, 5), 'mejoran': gg, 'empeoran': ll, 'z': round(z, 2),
                     'filas_test_que_cambian_vs_base': n_base, 'filas_test_que_cambian_vs_ref': n_ref, 'resolucion_publico': (None if np.isnan(res) else round(res, 5)),
                     'veredicto': veredicto, 'publico': np.nan, 'notas': ''})
    res_df = pd.DataFrame(rows)
    print('\n=== CANDIDATOS ===')
    print(res_df[['candidato', 'cv', 'delta_cv', 'mejoran', 'empeoran', 'z', 'filas_test_que_cambian_vs_base', 'resolucion_publico', 'veredicto']].to_string(index=False))
    print('\n"resolucion_publico" = diferencia mínima (±) que el puntaje público podría distinguir del ruido con ese número de filas distintas.')
    res_df.to_csv(LOG, mode='a', header=not os.path.exists(LOG), index=False)
    print('registro actualizado:', LOG, '| archivos para enviar:', os.listdir(os.path.join(LAB_DIR, 'para_enviar')))
else:
    print('laboratorio apagado')

laboratorio apagado


## 12. Inspector de errores del train y probador de reglas (`RUN_INSPECTOR = True`)
Es la forma **válida** de "mirar los registros y decidir cambios": mirás ejemplos del **train** (donde conocés la respuesta), encontrás un patrón y lo escribís como una **regla** que se aplica **automáticamente** a todas las filas, train y test. Nunca se decide a mano fila por fila en el test.

**Cómo está protegido contra el sobreajuste**
- `inspeccionar` solo muestra filas de los folds 0-2. Los folds 3-4 quedan como **confirmación**: una regla solo se acepta si mejora allí, con significancia.
- Cada regla que probás se anota en `laboratorio/reglas_probadas.csv`. **Cuantas más pruebes, más exigente se vuelve el umbral** (corrección de Bonferroni): probar 50 reglas y quedarte con la que "funcionó" es sobreajustar.
- `aplicar_reglas` genera un CSV candidato solo si el conjunto de reglas mejora el CV con evidencia suficiente.

**Uso**
```python
inspeccionar(40, banda=(0.15, 0.85))                                   # mirar filas dudosas del train
inspeccionar(40, solo_errores=True, query="titulo.str.contains('lote')")  # filtrar con pandas.query
probar_regla('lotes usados', lambda d: d['titulo'].str.contains(r'\blote\b'), 'used')
aplicar_reglas([('lotes usados', lambda d: d['titulo'].str.contains(r'\blote\b'), 'used')])
```
Columnas disponibles en las reglas: `titulo` (normalizado), `titulo_orig`, `garantia`, `tipo_garantia`, `categoria`, `vendedor`, `precio`, `n_vend`, `p` (probabilidad actual), `pred`.

In [14]:
if RUN_INSPECTOR:
    from scipy.stats import norm
    os.makedirs(LAB_DIR, exist_ok=True)
    Sa_, Sb_ = get_sources(); cnt_all = pd.concat([train['seller_id'], test['seller_id']]).value_counts()
    def _frame(raw, S, p):
        return pd.DataFrame({'titulo': S['title'].values, 'titulo_orig': raw['title'].values, 'garantia': raw['warranty'].fillna('').astype(str).values,
                             'tipo_garantia': _warr_bucket(S), 'categoria': raw['category_id'].values, 'vendedor': raw['seller_id'].values,
                             'precio': raw['price'].values, 'n_vend': raw['seller_id'].map(cnt_all).values, 'p': p, 'pred': (p > .5).astype(int)})
    INSP = _frame(train, Sa_, o_final); INSP['y'] = yv.values; INSP['ok'] = INSP['pred'] == INSP['y']
    INSP_T = _frame(test, Sb_, t_final)             # solo sirve para APLICAR las reglas; no se mira para decidir nada
    FOLD = np.zeros(ntr, int)
    for k, (_, va_i) in enumerate(skf.split(np.zeros(ntr), yv)): FOLD[va_i] = k
    RULES_LOG = os.path.join(LAB_DIR, 'reglas_probadas.csv')

    def inspeccionar(n=40, banda=None, solo_errores=False, query=None, seed=0):
        d = INSP[FOLD < 3]                           # folds 3-4 quedan ocultos: son la confirmación de las reglas
        if banda is not None: d = d[(d['p'] > banda[0]) & (d['p'] < banda[1])]
        if query: d = d.query(query, engine='python')
        if solo_errores: d = d[~d['ok']]
        d = d.sample(min(n, len(d)), random_state=seed)
        print(f'{len(d)} filas (de {int((FOLD < 3).sum())} visibles); columnas: acierto = pred == y real')
        print(d[['titulo_orig', 'garantia', 'categoria', 'n_vend', 'precio', 'p', 'y', 'ok']].round(3).to_string(max_colwidth=65))

    def _stats(new_pred):
        ok_new = (new_pred == INSP['y'].values); ok_old = INSP['ok'].values; gain, loss = ok_new & ~ok_old, ~ok_new & ok_old
        c = FOLD >= 3
        return dict(gd=int(gain[~c].sum()), ld=int(loss[~c].sum()), gc=int(gain[c].sum()), lc=int(loss[c].sum()), delta=ok_new.mean() - ok_old.mean(),
                    ok_new=ok_new)
    def _z(g, l): return (g - l) / np.sqrt(g + l) if g + l else 0.0

    def probar_regla(nombre, fn, etiqueta):
        lab = int(etiqueta == classes[-1]); m = np.asarray(fn(INSP), dtype=bool); mt = np.asarray(fn(INSP_T), dtype=bool)
        st = _stats(np.where(m, lab, INSP['pred'].values)); z_conf = _z(st['gc'], st['lc'])
        prev = pd.read_csv(RULES_LOG) if os.path.exists(RULES_LOG) else pd.DataFrame(); k = len(prev) + 1
        z_req = norm.isf(0.025 / k); acept = (st['gd'] - st['ld'] > 0) and z_conf >= z_req
        cambian_test = int((mt & (INSP_T['pred'].values != lab)).sum())
        print(f"regla '{nombre}' -> {etiqueta}: coincide en {int(m.sum())} filas del train ({int(mt.sum())} del test; cambiaría {cambian_test} predicciones del test)")
        print(f"  desarrollo (folds 0-2): mejoran {st['gd']}, empeoran {st['ld']} | confirmación (folds 3-4): mejoran {st['gc']}, empeoran {st['lc']}, z = {z_conf:.2f}")
        print(f"  reglas probadas hasta ahora (contando esta): {k} -> z requerido en la confirmación: {z_req:.2f}  =>  {'ACEPTADA' if acept else 'rechazada'}")
        pd.DataFrame([{'fecha': time.strftime('%Y%m%d_%H%M'), 'regla': nombre, 'etiqueta': etiqueta, 'filas_train': int(m.sum()), 'mejoran_dev': st['gd'], 'empeoran_dev': st['ld'],
                       'mejoran_conf': st['gc'], 'empeoran_conf': st['lc'], 'z_conf': round(z_conf, 2), 'z_requerido': round(z_req, 2), 'aceptada': bool(acept)}]
                     ).to_csv(RULES_LOG, mode='a', header=not os.path.exists(RULES_LOG), index=False)
        return acept

    def aplicar_reglas(reglas):
        pred_tr = INSP['pred'].values.copy(); pred_te = INSP_T['pred'].values.copy()
        for nombre, fn, etiqueta in reglas:
            lab = int(etiqueta == classes[-1]); pred_tr[np.asarray(fn(INSP), bool)] = lab; pred_te[np.asarray(fn(INSP_T), bool)] = lab
        st = _stats(pred_tr); z_all = _z(st['gd'] + st['gc'], st['ld'] + st['lc']); z_conf = _z(st['gc'], st['lc'])
        k_total = max(len(pd.read_csv(RULES_LOG)), 1) if os.path.exists(RULES_LOG) else 1; z_req = norm.isf(0.025 / k_total)
        cambian = int((pred_te != INSP_T['pred'].values).sum()); cv_new = st['ok_new'].mean()
        print(f"{len(reglas)} reglas | CV {INSP['ok'].mean():.5f} -> {cv_new:.5f} (diferencia {st['delta']:+.5f}) | confirmación: z = {z_conf:.2f} (requerido {z_req:.2f}) | filas del test que cambian: {cambian}")
        if st['delta'] >= LAB_MIN_DELTA and z_conf >= z_req:
            os.makedirs(os.path.join(LAB_DIR, 'para_enviar'), exist_ok=True)
            f = os.path.join(LAB_DIR, 'para_enviar', f"{time.strftime('%Y%m%d_%H%M')}_reglas_cv{cv_new:.4f}.csv")
            pd.DataFrame({'ID': range(1, nte + 1), TARGET: [classes[i] for i in pred_te]}).to_csv(f, index=False); print('ENVIAR -> guardado:', f)
        else: print('NO enviar: no hay evidencia suficiente de mejora.')
    print('inspector listo: inspeccionar(), probar_regla(), aplicar_reglas()')
else:
    print('inspector apagado')

inspector apagado
